# Fair benchmark Farooq-style (autocontido)

Notebook **independente** (local ou Kaggle): instala dependências, reconstrói o pacote `qml_air_quality` embutido, baixa Beijing/Aotizhongxin e executa o protocolo justo.

- purga temporal 24h + P90 no treino purgado
- val/teste fixos; só o treino varia por semente
- Grupo A (8D) + Grupo B (PCA-2 pareado clássicos/QSVM)
- Q01–Q04; limiar F2 na validação; block bootstrap
- plots em `artifacts/.../plots/`

**Kaggle:** Internet **ON** · Accelerator **None** (CPU)

| Modo | Treino | Val/Teste | Seeds | Tempo típico |
|---|---:|---:|---|---|
| `small` | 80 | 40/40 | 1 | ~5–20 min |
| `medium` | 200 | 100/150 | 5 | ~1–3 h |
| `large` | 200+500 | 200/300 | 5→10 | várias horas |



## 1. Instalação


In [ ]:
%pip install -q qiskit qiskit-machine-learning qiskit-aer scikit-learn pandas numpy matplotlib pyyaml joblib scipy


## 2. Ambiente + pacote embutido


In [ ]:
from __future__ import annotations

import json
import logging
import sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working/fair_benchmark") if ON_KAGGLE else Path.cwd() / "artifacts" / "fair_benchmark_nb"
WORK.mkdir(parents=True, exist_ok=True)
PKG_ROOT = WORK / "src"
PKG_ROOT.mkdir(parents=True, exist_ok=True)

# Pacote embutido (gerado a partir do repositório)
PACKAGE_FILES = json.loads('{"__init__.py": "\\"\\"\\"Helpers for the notebook PoC: classical vs QSVM on Beijing PM2.5 extremes.\\"\\"\\"\\n\\n__version__ = \\"0.1.0\\"\\n", "config.py": "\\"\\"\\"Load YAML configuration.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport yaml\\n\\n\\ndef project_root() -> Path:\\n    \\"\\"\\"Return repository root (parent of config/).\\"\\"\\"\\n    return Path(__file__).resolve().parents[2]\\n\\n\\ndef load_config(path: str | Path | None = None) -> dict[str, Any]:\\n    \\"\\"\\"Load PoC YAML config; default is config/poc.yaml at repo root.\\"\\"\\"\\n    cfg_path = Path(path) if path else project_root() / \\"config\\" / \\"poc.yaml\\"\\n    with cfg_path.open(encoding=\\"utf-8\\") as f:\\n        return yaml.safe_load(f)\\n", "data.py": "\\"\\"\\"Download and load Beijing Multi-Site Air Quality (UCI).\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport logging\\nimport zipfile\\nfrom io import BytesIO\\nfrom pathlib import Path\\nfrom typing import Any\\nfrom urllib.request import urlopen\\n\\nimport pandas as pd\\n\\nfrom qml_air_quality.config import project_root\\n\\nlogger = logging.getLogger(__name__)\\n\\nUCI_ZIP_URLS = [\\n    # Prefer the historical PRSA archive (full per-station CSVs).\\n    \\"https://archive.ics.uci.edu/ml/machine-learning-databases/00501/PRSA2017_Data_20130301-20170228.zip\\",\\n    \\"https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip\\",\\n]\\n\\n\\ndef _checksum(path: Path) -> str:\\n    h = hashlib.sha256()\\n    with path.open(\\"rb\\") as f:\\n        for chunk in iter(lambda: f.read(1 << 20), b\\"\\"):\\n            h.update(chunk)\\n    return h.hexdigest()\\n\\n\\ndef _download_bytes(urls: list[str]) -> tuple[bytes, str]:\\n    last_err: Exception | None = None\\n    for url in urls:\\n        try:\\n            logger.info(\\"Downloading %s\\", url)\\n            with urlopen(url, timeout=120) as resp:\\n                data = resp.read()\\n            if data:\\n                return data, url\\n        except Exception as exc:  # noqa: BLE001 — try next mirror\\n            last_err = exc\\n            logger.warning(\\"Download failed for %s: %s\\", url, exc)\\n    raise RuntimeError(f\\"Could not download UCI dataset: {last_err}\\")\\n\\n\\ndef _extract_station_csv(zip_bytes: bytes, station: str) -> pd.DataFrame:\\n    with zipfile.ZipFile(BytesIO(zip_bytes)) as zf:\\n        members = [n for n in zf.namelist() if n.lower().endswith(\\".csv\\")]\\n        match = [n for n in members if station.lower() in Path(n).name.lower()]\\n        if not match:\\n            raise FileNotFoundError(\\n                f\\"No CSV for station={station!r} in zip. Available: {members}\\"\\n            )\\n        with zf.open(match[0]) as f:\\n            return pd.read_csv(f)\\n\\n\\ndef download_dataset(\\n    dataset_id: int = 501,\\n    station: str = \\"Aotizhongxin\\",\\n    raw_dir: str | Path | None = None,\\n) -> Path:\\n    \\"\\"\\"Download UCI zip (if needed) and save station CSV to data/raw.\\n\\n    Note: ucimlrepo does not expose dataset 501 for Python import, so we\\n    fetch the official zip mirrors directly.\\n    \\"\\"\\"\\n    _ = dataset_id  # kept for config compatibility\\n    root = project_root()\\n    raw = Path(raw_dir) if raw_dir else root / \\"data\\" / \\"raw\\"\\n    raw.mkdir(parents=True, exist_ok=True)\\n\\n    out_csv = raw / f\\"{station.lower()}_air_quality.csv\\"\\n    meta_path = raw / f\\"{station.lower()}_download_meta.json\\"\\n    zip_path = raw / \\"beijing_multi_site_air_quality.zip\\"\\n\\n    if out_csv.exists() and meta_path.exists():\\n        meta = json.loads(meta_path.read_text(encoding=\\"utf-8\\"))\\n        if meta.get(\\"sha256\\") == _checksum(out_csv):\\n            logger.info(\\"Raw data already present and checksum OK: %s\\", out_csv)\\n            return out_csv\\n\\n    def _zip_has_station(data: bytes) -> bool:\\n        try:\\n            with zipfile.ZipFile(BytesIO(data)) as zf:\\n                return any(\\n                    station.lower() in Path(n).name.lower() and n.lower().endswith(\\".csv\\")\\n                    for n in zf.namelist()\\n                )\\n        except zipfile.BadZipFile:\\n            return False\\n\\n    zip_bytes: bytes | None = None\\n    source = \\"\\"\\n    if zip_path.exists():\\n        candidate = zip_path.read_bytes()\\n        if _zip_has_station(candidate):\\n            zip_bytes = candidate\\n            source = str(zip_path)\\n        else:\\n            logger.warning(\\"Cached zip missing station CSV; re-downloading\\")\\n\\n    if zip_bytes is None:\\n        zip_bytes, source = _download_bytes(UCI_ZIP_URLS)\\n        if not _zip_has_station(zip_bytes):\\n            # Try remaining mirrors explicitly if first zip is the wrong bundle\\n            for url in UCI_ZIP_URLS:\\n                if url == source:\\n                    continue\\n                try:\\n                    with urlopen(url, timeout=120) as resp:\\n                        alt = resp.read()\\n                    if _zip_has_station(alt):\\n                        zip_bytes, source = alt, url\\n                        break\\n                except Exception as exc:  # noqa: BLE001\\n                    logger.warning(\\"Mirror failed %s: %s\\", url, exc)\\n        zip_path.write_bytes(zip_bytes)\\n\\n    station_df = _extract_station_csv(zip_bytes, station)\\n    if station_df.empty:\\n        raise ValueError(f\\"No rows for station={station!r}\\")\\n\\n    station_df.to_csv(out_csv, index=False)\\n    meta: dict[str, Any] = {\\n        \\"dataset_id\\": dataset_id,\\n        \\"station\\": station,\\n        \\"source\\": source,\\n        \\"n_rows\\": len(station_df),\\n        \\"sha256\\": _checksum(out_csv),\\n        \\"path\\": str(out_csv.relative_to(root)),\\n    }\\n    meta_path.write_text(json.dumps(meta, indent=2), encoding=\\"utf-8\\")\\n    logger.info(\\"Saved %s rows to %s\\", len(station_df), out_csv)\\n    return out_csv\\n\\n\\ndef build_timestamp(df: pd.DataFrame) -> pd.DataFrame:\\n    \\"\\"\\"Build monotonic timestamp from year/month/day/hour.\\"\\"\\"\\n    out = df.copy()\\n    required = [\\"year\\", \\"month\\", \\"day\\", \\"hour\\"]\\n    rename = {}\\n    for c in required:\\n        for col in out.columns:\\n            if col.lower() == c and col != c:\\n                rename[col] = c\\n    if rename:\\n        out = out.rename(columns=rename)\\n    missing = [c for c in required if c not in out.columns]\\n    if missing:\\n        raise KeyError(f\\"Missing time columns: {missing}\\")\\n\\n    out[\\"timestamp\\"] = pd.to_datetime(\\n        {\\n            \\"year\\": out[\\"year\\"].astype(int),\\n            \\"month\\": out[\\"month\\"].astype(int),\\n            \\"day\\": out[\\"day\\"].astype(int),\\n            \\"hour\\": out[\\"hour\\"].astype(int),\\n        }\\n    )\\n    out = out.dropna(subset=[\\"timestamp\\"])\\n    out = out.sort_values(\\"timestamp\\").drop_duplicates(subset=[\\"timestamp\\"], keep=\\"first\\")\\n    out = out.reset_index(drop=True)\\n    if not out[\\"timestamp\\"].is_monotonic_increasing:\\n        raise ValueError(\\"timestamp is not monotonic increasing after sort\\")\\n    return out\\n\\n\\ndef load_station_csv(path: str | Path) -> pd.DataFrame:\\n    \\"\\"\\"Load station CSV and attach timestamp.\\"\\"\\"\\n    df = pd.read_csv(path)\\n    return build_timestamp(df)\\n\\n\\ndef missing_report(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:\\n    \\"\\"\\"Per-column missing counts and rates.\\"\\"\\"\\n    rows = []\\n    n = len(df)\\n    for c in columns:\\n        if c not in df.columns:\\n            continue\\n        miss = int(df[c].isna().sum())\\n        rows.append({\\"column\\": c, \\"missing\\": miss, \\"rate\\": miss / n if n else 0.0})\\n    return pd.DataFrame(rows)\\n", "features.py": "\\"\\"\\"Feature engineering and target creation (no future leakage).\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport numpy as np\\nimport pandas as pd\\n\\n\\ndef add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:\\n    \\"\\"\\"Cyclical hour/month encodings from timestamp.\\"\\"\\"\\n    out = df.copy()\\n    hour = out[\\"timestamp\\"].dt.hour\\n    month = out[\\"timestamp\\"].dt.month\\n    out[\\"hour_sin\\"] = np.sin(2 * np.pi * hour / 24)\\n    out[\\"hour_cos\\"] = np.cos(2 * np.pi * hour / 24)\\n    out[\\"month_sin\\"] = np.sin(2 * np.pi * month / 12)\\n    out[\\"month_cos\\"] = np.cos(2 * np.pi * month / 12)\\n    return out\\n\\n\\ndef add_lag_and_rolling(\\n    df: pd.DataFrame,\\n    columns: list[str],\\n    lag_hours: list[int],\\n    rolling_windows: list[int],\\n) -> pd.DataFrame:\\n    \\"\\"\\"Causal lags and rolling stats (past-only windows).\\"\\"\\"\\n    out = df.copy()\\n    for col in columns:\\n        if col not in out.columns:\\n            raise KeyError(col)\\n        for lag in lag_hours:\\n            out[f\\"{col}_lag_{lag}\\"] = out[col].shift(lag)\\n        for w in rolling_windows:\\n            # shift(1) so window ends at t-1 … actually plan says value at t is allowed\\n            # rolling mean of last w hours including current: rolling(w) is causal\\n            out[f\\"{col}_roll_mean_{w}\\"] = out[col].rolling(window=w, min_periods=1).mean()\\n            out[f\\"{col}_roll_std_{w}\\"] = out[col].rolling(window=w, min_periods=1).std()\\n    return out\\n\\n\\ndef add_future_target(\\n    df: pd.DataFrame,\\n    target_column: str = \\"PM2.5\\",\\n    horizon_hours: int = 24,\\n) -> pd.DataFrame:\\n    \\"\\"\\"Add future_pm25 = target shifted -horizon (label only; not a feature).\\"\\"\\"\\n    out = df.copy()\\n    out[\\"feature_timestamp\\"] = out[\\"timestamp\\"]\\n    out[\\"future_pm25\\"] = out[target_column].shift(-horizon_hours)\\n    out[\\"target_timestamp\\"] = out[\\"timestamp\\"] + pd.Timedelta(hours=horizon_hours)\\n    return out\\n\\n\\ndef apply_causal_ffill(\\n    df: pd.DataFrame,\\n    columns: list[str],\\n    limit_hours: int = 3,\\n) -> pd.DataFrame:\\n    \\"\\"\\"Forward-fill limited to limit_hours (causal).\\"\\"\\"\\n    out = df.copy()\\n    for c in columns:\\n        if c in out.columns:\\n            out[c] = out[c].ffill(limit=limit_hours)\\n    return out\\n\\n\\ndef impute_with_train_medians(\\n    train: pd.DataFrame,\\n    *others: pd.DataFrame,\\n    columns: list[str],\\n) -> tuple[pd.DataFrame, ...]:\\n    \\"\\"\\"Fill remaining NaNs using medians computed only on train.\\"\\"\\"\\n    medians = {c: train[c].median() for c in columns if c in train.columns}\\n    result = []\\n    for part in (train, *others):\\n        out = part.copy()\\n        for c, med in medians.items():\\n            if c in out.columns:\\n                out[c] = out[c].fillna(med)\\n        result.append(out)\\n    return tuple(result)\\n\\n\\ndef make_binary_target(\\n    train: pd.DataFrame,\\n    validation: pd.DataFrame,\\n    test: pd.DataFrame,\\n    percentile: float = 0.90,\\n    source: str = \\"training_partition\\",\\n    horizon_hours: int = 24,\\n) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, Any]]:\\n    \\"\\"\\"Threshold from train future_pm25 only; attach target 0/1.\\"\\"\\"\\n    threshold = float(train[\\"future_pm25\\"].quantile(percentile))\\n    meta = {\\n        \\"target\\": \\"PM2.5\\",\\n        \\"horizon_hours\\": horizon_hours,\\n        \\"percentile\\": percentile,\\n        \\"threshold\\": threshold,\\n        \\"threshold_source\\": source,\\n        \\"source\\": source,\\n    }\\n\\n    def _apply(part: pd.DataFrame) -> pd.DataFrame:\\n        out = part.copy()\\n        out[\\"target\\"] = (out[\\"future_pm25\\"] >= threshold).astype(int)\\n        return out\\n\\n    return _apply(train), _apply(validation), _apply(test), meta\\n\\n\\ndef add_farooq_style_stats(\\n    df: pd.DataFrame,\\n    source_map: dict[str, str] | None = None,\\n    window: int = 24,\\n    min_periods: int = 12,\\n) -> tuple[pd.DataFrame, list[str]]:\\n    \\"\\"\\"Causal rolling min/max/median/variance in Farooq naming.\\n\\n    Example outputs: pm25_min, pm25_max, pm25_median, pm25_variance,\\n    temperature_min, temperature_max, temperature_median, temperature_variance.\\n    \\"\\"\\"\\n    source_map = source_map or {\\"PM2.5\\": \\"pm25\\", \\"TEMP\\": \\"temperature\\"}\\n    out = df.copy()\\n    feat_cols: list[str] = []\\n    for src, prefix in source_map.items():\\n        if src not in out.columns:\\n            raise KeyError(f\\"Missing source column {src!r}\\")\\n        roll = out[src].rolling(window=window, min_periods=min_periods)\\n        mapping = {\\n            f\\"{prefix}_min\\": roll.min(),\\n            f\\"{prefix}_max\\": roll.max(),\\n            f\\"{prefix}_median\\": roll.median(),\\n            f\\"{prefix}_variance\\": roll.var(),\\n        }\\n        for name, series in mapping.items():\\n            out[name] = series\\n            feat_cols.append(name)\\n    return out, feat_cols\\n\\n\\n# EPA PM2.5 24h concentration breakpoints (µg/m³) → AQI category names.\\n# Used to mirror Farooq-style \\"Good / Moderate / …\\" bands on concentration.\\nEPA_PM25_BREAKPOINTS: list[tuple[float, str]] = [\\n    (12.0, \\"Good\\"),\\n    (35.4, \\"Moderate\\"),\\n    (55.4, \\"Unhealthy_Sensitive\\"),\\n    (150.4, \\"Unhealthy\\"),\\n    (250.4, \\"Very_Unhealthy\\"),\\n    (float(\\"inf\\"), \\"Hazardous\\"),\\n]\\n\\n\\ndef pm25_to_aqi_band(pm25: float | pd.Series) -> str | pd.Series:\\n    \\"\\"\\"Map PM2.5 concentration (µg/m³) to EPA-style band name.\\"\\"\\"\\n    if isinstance(pm25, pd.Series):\\n        out = pd.Series(index=pm25.index, dtype=object)\\n        out[:] = \\"Hazardous\\"\\n        prev = -float(\\"inf\\")\\n        for hi, name in EPA_PM25_BREAKPOINTS:\\n            mask = (pm25 > prev) & (pm25 <= hi)\\n            out.loc[mask] = name\\n            prev = hi\\n        out.loc[pm25.isna()] = pd.NA\\n        return out\\n\\n    if pm25 is None or (isinstance(pm25, float) and np.isnan(pm25)):\\n        return pd.NA  # type: ignore[return-value]\\n    prev = -float(\\"inf\\")\\n    for hi, name in EPA_PM25_BREAKPOINTS:\\n        if prev < float(pm25) <= hi:\\n            return name\\n        prev = hi\\n    return \\"Hazardous\\"\\n\\n\\ndef add_aqi_targets(\\n    train: pd.DataFrame,\\n    validation: pd.DataFrame,\\n    test: pd.DataFrame,\\n    value_col: str = \\"future_pm25\\",\\n) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, Any]]:\\n    \\"\\"\\"Attach AQI band labels and binary targets derived from value_col.\\n\\n    Targets:\\n      - aqi_band: Good / Moderate / Unhealthy_Sensitive / …\\n      - aqi_bad: 1 if band worse than Moderate (Unhealthy_Sensitive+)\\n      - aqi_good_vs_moderate: 1=Moderate, 0=Good (only meaningful on those two bands)\\n    \\"\\"\\"\\n    meta = {\\n        \\"value_col\\": value_col,\\n        \\"breakpoints_ugm3\\": [\\n            {\\"max\\": 12.0, \\"band\\": \\"Good\\"},\\n            {\\"max\\": 35.4, \\"band\\": \\"Moderate\\"},\\n            {\\"max\\": 55.4, \\"band\\": \\"Unhealthy_Sensitive\\"},\\n            {\\"max\\": 150.4, \\"band\\": \\"Unhealthy\\"},\\n            {\\"max\\": 250.4, \\"band\\": \\"Very_Unhealthy\\"},\\n            {\\"max\\": None, \\"band\\": \\"Hazardous\\"},\\n        ],\\n        \\"aqi_bad_definition\\": \\"1 if band not in {Good, Moderate}\\",\\n        \\"aqi_good_vs_moderate_definition\\": \\"0=Good, 1=Moderate (rows outside dropped by caller)\\",\\n    }\\n\\n    def _apply(part: pd.DataFrame) -> pd.DataFrame:\\n        out = part.copy()\\n        out[\\"aqi_band\\"] = pm25_to_aqi_band(out[value_col])\\n        out[\\"aqi_bad\\"] = (~out[\\"aqi_band\\"].isin([\\"Good\\", \\"Moderate\\"])).astype(int)\\n        # Farooq-like Good vs Moderate encoding (invalid outside those bands)\\n        out[\\"aqi_good_vs_moderate\\"] = np.where(\\n            out[\\"aqi_band\\"] == \\"Good\\",\\n            0,\\n            np.where(out[\\"aqi_band\\"] == \\"Moderate\\", 1, np.nan),\\n        )\\n        return out\\n\\n    return _apply(train), _apply(validation), _apply(test), meta\\n\\n\\ndef feature_columns(df: pd.DataFrame) -> list[str]:\\n    \\"\\"\\"Numeric feature columns excluding target/future/id leakage names.\\"\\"\\"\\n    banned_substrings = (\\"future\\", \\"target\\", \\"t_plus_24\\")\\n    ban_exact = {\\n        \\"timestamp\\",\\n        \\"feature_timestamp\\",\\n        \\"target_timestamp\\",\\n        \\"year\\",\\n        \\"month\\",\\n        \\"day\\",\\n        \\"hour\\",\\n        \\"station\\",\\n        \\"No\\",\\n        \\"wd\\",\\n        \\"future_pm25\\",\\n        \\"target\\",\\n    }\\n    cols = []\\n    for c in df.columns:\\n        if c in ban_exact:\\n            continue\\n        cl = c.lower()\\n        if any(b in cl for b in banned_substrings):\\n            continue\\n        if not pd.api.types.is_numeric_dtype(df[c]):\\n            continue\\n        cols.append(c)\\n    return cols\\n\\n\\ndef save_target_metadata(meta: dict[str, Any], path: str | Path) -> None:\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    path.write_text(json.dumps(meta, indent=2), encoding=\\"utf-8\\")\\n", "split.py": "\\"\\"\\"Temporal split and stratified subsample for quantum comparison.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.model_selection import train_test_split\\n\\n\\ndef temporal_split(\\n    df: pd.DataFrame,\\n    train_fraction: float = 0.60,\\n    validation_fraction: float = 0.20,\\n    test_fraction: float = 0.20,\\n) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\\n    \\"\\"\\"Chronological split; rows with NaN future target are dropped first.\\"\\"\\"\\n    if not np.isclose(train_fraction + validation_fraction + test_fraction, 1.0):\\n        raise ValueError(\\"split fractions must sum to 1\\")\\n\\n    data = df.dropna(subset=[\\"future_pm25\\"]).sort_values(\\"timestamp\\").reset_index(drop=True)\\n    n = len(data)\\n    i_train = int(n * train_fraction)\\n    i_val = i_train + int(n * validation_fraction)\\n\\n    train = data.iloc[:i_train].copy()\\n    validation = data.iloc[i_train:i_val].copy()\\n    test = data.iloc[i_val:].copy()\\n\\n    if len(train) == 0 or len(validation) == 0 or len(test) == 0:\\n        raise ValueError(\\"empty partition after temporal split\\")\\n\\n    if not (train[\\"timestamp\\"].max() < validation[\\"timestamp\\"].min()):\\n        raise ValueError(\\"train/validation temporal overlap\\")\\n    if not (validation[\\"timestamp\\"].max() < test[\\"timestamp\\"].min()):\\n        raise ValueError(\\"validation/test temporal overlap\\")\\n\\n    return train, validation, test\\n\\n\\ndef apply_temporal_purge(\\n    train: pd.DataFrame,\\n    validation: pd.DataFrame,\\n    test: pd.DataFrame,\\n    purge_hours: int = 24,\\n) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict[str, Any]]:\\n    \\"\\"\\"Remove border rows whose target_timestamp leaks into the next partition.\\n\\n    Keeps train rows with target_timestamp < validation feature start, and\\n    validation rows with target_timestamp < test feature start.\\n    \\"\\"\\"\\n    if \\"target_timestamp\\" not in train.columns:\\n        raise KeyError(\\"target_timestamp is required for temporal purge\\")\\n\\n    validation_start = validation[\\"timestamp\\"].min()\\n    test_start = test[\\"timestamp\\"].min()\\n\\n    n_train_before = len(train)\\n    n_val_before = len(validation)\\n\\n    train_p = train[train[\\"target_timestamp\\"] < validation_start].copy()\\n    validation_p = validation[validation[\\"target_timestamp\\"] < test_start].copy()\\n    test_p = test.copy()\\n\\n    if len(train_p) == 0 or len(validation_p) == 0 or len(test_p) == 0:\\n        raise ValueError(\\"empty partition after temporal purge\\")\\n\\n    if train_p[\\"target_timestamp\\"].max() >= validation_start:\\n        raise AssertionError(\\"train target_timestamp leaks into validation\\")\\n    if validation_p[\\"target_timestamp\\"].max() >= test_start:\\n        raise AssertionError(\\"validation target_timestamp leaks into test\\")\\n\\n    meta = {\\n        \\"purge_hours\\": purge_hours,\\n        \\"validation_start\\": str(validation_start),\\n        \\"test_start\\": str(test_start),\\n        \\"train_rows_before\\": n_train_before,\\n        \\"train_rows_after\\": len(train_p),\\n        \\"train_rows_removed\\": n_train_before - len(train_p),\\n        \\"validation_rows_before\\": n_val_before,\\n        \\"validation_rows_after\\": len(validation_p),\\n        \\"validation_rows_removed\\": n_val_before - len(validation_p),\\n        \\"test_rows\\": len(test_p),\\n        \\"train_target_timestamp_max\\": str(train_p[\\"target_timestamp\\"].max()),\\n        \\"validation_target_timestamp_max\\": str(validation_p[\\"target_timestamp\\"].max()),\\n    }\\n    return train_p, validation_p, test_p, meta\\n\\n\\ndef stratified_subsample(\\n    train: pd.DataFrame,\\n    validation: pd.DataFrame,\\n    test: pd.DataFrame,\\n    train_size: int = 500,\\n    validation_size: int = 150,\\n    test_size: int = 200,\\n    seed: int = 42,\\n) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:\\n    \\"\\"\\"Stratified subsample by target; returns subsets + index table.\\"\\"\\"\\n\\n    def _sample(part: pd.DataFrame, size: int, name: str) -> pd.DataFrame:\\n        size = min(size, len(part))\\n        if size < len(part):\\n            idx, _ = train_test_split(\\n                part.index,\\n                train_size=size,\\n                stratify=part[\\"target\\"],\\n                random_state=seed,\\n            )\\n            out = part.loc[idx].sort_values(\\"timestamp\\").copy()\\n        else:\\n            out = part.copy()\\n        out = out.reset_index(names=\\"original_index\\")\\n        out[\\"partition\\"] = name\\n        return out\\n\\n    tr = _sample(train, train_size, \\"train\\")\\n    va = _sample(validation, validation_size, \\"validation\\")\\n    te = _sample(test, test_size, \\"test\\")\\n\\n    index_table = pd.concat(\\n        [\\n            tr[[\\"original_index\\", \\"timestamp\\", \\"target\\", \\"partition\\"]],\\n            va[[\\"original_index\\", \\"timestamp\\", \\"target\\", \\"partition\\"]],\\n            te[[\\"original_index\\", \\"timestamp\\", \\"target\\", \\"partition\\"]],\\n        ],\\n        ignore_index=True,\\n    )\\n    return tr, va, te, index_table\\n\\n\\ndef save_sample_indices(index_table: pd.DataFrame, path: str | Path) -> None:\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    index_table.to_csv(path, index=False)\\n", "sample_registry.py": "\\"\\"\\"Fixed validation/test indices; train subsample varies by seed.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport pandas as pd\\nfrom sklearn.model_selection import train_test_split\\n\\n\\ndef _stratified_indices(\\n    part: pd.DataFrame,\\n    size: int,\\n    seed: int,\\n) -> pd.Index:\\n    size = min(size, len(part))\\n    if size >= len(part):\\n        return part.index\\n    idx, _ = train_test_split(\\n        part.index,\\n        train_size=size,\\n        stratify=part[\\"target\\"],\\n        random_state=seed,\\n    )\\n    return pd.Index(idx)\\n\\n\\ndef _rows_from_indices(part: pd.DataFrame, indices: pd.Index, partition: str) -> pd.DataFrame:\\n    out = part.loc[indices].sort_values(\\"timestamp\\").copy()\\n    out = out.reset_index(names=\\"original_index\\")\\n    out[\\"partition\\"] = partition\\n    return out\\n\\n\\ndef build_fixed_evaluation_sets(\\n    validation: pd.DataFrame,\\n    test: pd.DataFrame,\\n    validation_size: int = 200,\\n    test_size: int = 300,\\n    evaluation_seed: int = 2026,\\n) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:\\n    \\"\\"\\"Select validation and test once; reuse across all training seeds.\\"\\"\\"\\n    val_idx = _stratified_indices(validation, validation_size, evaluation_seed)\\n    test_idx = _stratified_indices(test, test_size, evaluation_seed + 1)\\n\\n    va = _rows_from_indices(validation, val_idx, \\"validation\\")\\n    te = _rows_from_indices(test, test_idx, \\"test\\")\\n    meta = {\\n        \\"evaluation_seed\\": evaluation_seed,\\n        \\"validation_size\\": len(va),\\n        \\"test_size\\": len(te),\\n        \\"validation_indices\\": va[\\"original_index\\"].tolist(),\\n        \\"test_indices\\": te[\\"original_index\\"].tolist(),\\n        \\"fixed_validation\\": True,\\n        \\"fixed_test\\": True,\\n    }\\n    return va, te, meta\\n\\n\\ndef sample_train_for_seed(\\n    train: pd.DataFrame,\\n    train_size: int,\\n    seed: int,\\n) -> pd.DataFrame:\\n    \\"\\"\\"Stratified training subsample that varies with seed.\\"\\"\\"\\n    idx = _stratified_indices(train, train_size, seed)\\n    return _rows_from_indices(train, idx, \\"train\\")\\n\\n\\ndef save_partition_csv(df: pd.DataFrame, path: str | Path) -> None:\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    cols = [c for c in [\\"original_index\\", \\"timestamp\\", \\"target\\", \\"partition\\"] if c in df.columns]\\n    extra = [c for c in df.columns if c not in cols]\\n    df[cols + extra].to_csv(path, index=False)\\n\\n\\ndef load_partition_indices(path: str | Path) -> list[Any]:\\n    df = pd.read_csv(path)\\n    return df[\\"original_index\\"].tolist()\\n\\n\\nclass SampleRegistry:\\n    \\"\\"\\"Persist fixed val/test and per-seed train indices under artifacts.\\"\\"\\"\\n\\n    def __init__(self, root: str | Path):\\n        self.root = Path(root)\\n        self.root.mkdir(parents=True, exist_ok=True)\\n\\n    def save_fixed(\\n        self,\\n        validation: pd.DataFrame,\\n        test: pd.DataFrame,\\n    ) -> None:\\n        save_partition_csv(validation, self.root / \\"validation_fixed.csv\\")\\n        save_partition_csv(test, self.root / \\"test_fixed.csv\\")\\n\\n    def save_train(self, train: pd.DataFrame, seed: int, train_size: int | None = None) -> None:\\n        suffix = f\\"train_seed_{seed}\\"\\n        if train_size is not None:\\n            suffix = f\\"train_seed_{seed}_n{train_size}\\"\\n        save_partition_csv(train, self.root / f\\"{suffix}.csv\\")\\n\\n    def load_fixed_indices(self) -> tuple[list[Any], list[Any]]:\\n        val = load_partition_indices(self.root / \\"validation_fixed.csv\\")\\n        test = load_partition_indices(self.root / \\"test_fixed.csv\\")\\n        return val, test\\n", "paired_representation.py": "\\"\\"\\"Paired full-8D and PCA-2 + angular representations (train-only fit).\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass, field\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport joblib\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.decomposition import PCA\\nfrom sklearn.impute import SimpleImputer\\nfrom sklearn.pipeline import Pipeline\\nfrom sklearn.preprocessing import StandardScaler\\n\\nfrom qml_air_quality.preprocessing.angular_scaling import (\\n    AngularScaler,\\n    make_angular_scaler,\\n)\\n\\n\\n@dataclass\\nclass FittedTransform:\\n    \\"\\"\\"Tracks that transformers were fit only on the train partition.\\"\\"\\"\\n\\n    name: str\\n    fit_partition: str = \\"train\\"\\n    transformer: Any = None\\n\\n\\n@dataclass\\nclass PairedRepresentation:\\n    \\"\\"\\"Full 8D and PCA-2 embeddings for classical/quantum fair comparison.\\"\\"\\"\\n\\n    feature_cols: list[str]\\n    X_train_full: np.ndarray\\n    X_val_full: np.ndarray\\n    X_test_full: np.ndarray\\n    X_train_pca: np.ndarray\\n    X_val_pca: np.ndarray\\n    X_test_pca: np.ndarray\\n    full_pipeline: Pipeline\\n    pca_pipeline: Pipeline\\n    fit_records: list[FittedTransform] = field(default_factory=list)\\n    explained_variance_ratio: np.ndarray | None = None\\n\\n    def angular(\\n        self,\\n        scaler_name: str,\\n        seed: int = 42,\\n    ) -> tuple[np.ndarray, np.ndarray, np.ndarray, AngularScaler, FittedTransform]:\\n        ang = make_angular_scaler(scaler_name, seed=seed)\\n        X_tr = ang.fit_transform(self.X_train_pca)\\n        X_va = ang.transform(self.X_val_pca)\\n        X_te = ang.transform(self.X_test_pca)\\n        record = FittedTransform(name=f\\"angular_{scaler_name}\\", fit_partition=\\"train\\", transformer=ang)\\n        return X_tr, X_va, X_te, ang, record\\n\\n\\ndef build_full_pipeline() -> Pipeline:\\n    return Pipeline(\\n        steps=[\\n            (\\"imputer\\", SimpleImputer(strategy=\\"median\\")),\\n            (\\"scaler\\", StandardScaler()),\\n        ]\\n    )\\n\\n\\ndef build_pca_pipeline(n_components: int = 2, seed: int = 42) -> Pipeline:\\n    return Pipeline(\\n        steps=[\\n            (\\"imputer\\", SimpleImputer(strategy=\\"median\\")),\\n            (\\"scaler\\", StandardScaler()),\\n            (\\"pca\\", PCA(n_components=n_components, random_state=seed)),\\n        ]\\n    )\\n\\n\\ndef fit_paired_representation(\\n    train: pd.DataFrame,\\n    validation: pd.DataFrame,\\n    test: pd.DataFrame,\\n    feature_cols: list[str],\\n    pca_components: int = 2,\\n    seed: int = 42,\\n) -> PairedRepresentation:\\n    \\"\\"\\"Fit imputer/scaler/PCA on train only; transform all partitions.\\"\\"\\"\\n    full_pipe = build_full_pipeline()\\n    pca_pipe = build_pca_pipeline(n_components=pca_components, seed=seed)\\n\\n    X_train_full = full_pipe.fit_transform(train[feature_cols])\\n    X_val_full = full_pipe.transform(validation[feature_cols])\\n    X_test_full = full_pipe.transform(test[feature_cols])\\n\\n    X_train_pca = pca_pipe.fit_transform(train[feature_cols])\\n    X_val_pca = pca_pipe.transform(validation[feature_cols])\\n    X_test_pca = pca_pipe.transform(test[feature_cols])\\n\\n    records = [\\n        FittedTransform(\\"full_imputer_scaler\\", \\"train\\", full_pipe),\\n        FittedTransform(\\"pca_pipeline\\", \\"train\\", pca_pipe),\\n    ]\\n    explained = pca_pipe.named_steps[\\"pca\\"].explained_variance_ratio_\\n    return PairedRepresentation(\\n        feature_cols=list(feature_cols),\\n        X_train_full=np.asarray(X_train_full, dtype=float),\\n        X_val_full=np.asarray(X_val_full, dtype=float),\\n        X_test_full=np.asarray(X_test_full, dtype=float),\\n        X_train_pca=np.asarray(X_train_pca, dtype=float),\\n        X_val_pca=np.asarray(X_val_pca, dtype=float),\\n        X_test_pca=np.asarray(X_test_pca, dtype=float),\\n        full_pipeline=full_pipe,\\n        pca_pipeline=pca_pipe,\\n        fit_records=records,\\n        explained_variance_ratio=explained,\\n    )\\n\\n\\ndef save_preprocessors(repr_: PairedRepresentation, directory: str | Path) -> None:\\n    directory = Path(directory)\\n    directory.mkdir(parents=True, exist_ok=True)\\n    joblib.dump(repr_.full_pipeline, directory / \\"full_pipeline.joblib\\")\\n    joblib.dump(repr_.pca_pipeline, directory / \\"pca_pipeline.joblib\\")\\n", "metrics.py": "\\"\\"\\"Evaluation metrics for binary extreme-event classification.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.metrics import (\\n    average_precision_score,\\n    balanced_accuracy_score,\\n    f1_score,\\n    fbeta_score,\\n    matthews_corrcoef,\\n    precision_score,\\n    recall_score,\\n    roc_auc_score,\\n)\\n\\n\\ndef binary_metrics(\\n    y_true: np.ndarray,\\n    y_pred: np.ndarray,\\n    y_proba: np.ndarray | None = None,\\n    model_name: str = \\"\\",\\n    training_seconds: float | None = None,\\n    inference_seconds: float | None = None,\\n) -> dict[str, Any]:\\n    \\"\\"\\"Compute primary (AUPRC) and secondary / operational metrics.\\"\\"\\"\\n    y_true = np.asarray(y_true)\\n    y_pred = np.asarray(y_pred)\\n    scores = y_proba if y_proba is not None else y_pred\\n\\n    tp = int(((y_pred == 1) & (y_true == 1)).sum())\\n    fp = int(((y_pred == 1) & (y_true == 0)).sum())\\n    fn = int(((y_pred == 0) & (y_true == 1)).sum())\\n    n = len(y_true)\\n    n_pos = int((y_true == 1).sum())\\n\\n    out: dict[str, Any] = {\\n        \\"model\\": model_name,\\n        \\"average_precision\\": float(average_precision_score(y_true, scores)),\\n        \\"balanced_accuracy\\": float(balanced_accuracy_score(y_true, y_pred)),\\n        \\"precision_extreme\\": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),\\n        \\"recall_extreme\\": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),\\n        \\"f1_extreme\\": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),\\n        \\"f2_extreme\\": float(fbeta_score(y_true, y_pred, beta=2, pos_label=1, zero_division=0)),\\n        \\"mcc\\": float(matthews_corrcoef(y_true, y_pred)),\\n        \\"false_alert_rate\\": float(fp / n) if n else float(\\"nan\\"),\\n        \\"missed_extreme_rate\\": float(fn / n_pos) if n_pos else float(\\"nan\\"),\\n        \\"false_positives\\": fp,\\n        \\"false_negatives\\": fn,\\n        \\"true_positives\\": tp,\\n        \\"n_predictions\\": n,\\n        \\"n_positives\\": n_pos,\\n    }\\n    if y_proba is not None and len(np.unique(y_true)) > 1:\\n        out[\\"auroc\\"] = float(roc_auc_score(y_true, y_proba))\\n    else:\\n        out[\\"auroc\\"] = float(\\"nan\\")\\n    if training_seconds is not None:\\n        out[\\"training_seconds\\"] = training_seconds\\n    if inference_seconds is not None:\\n        out[\\"inference_seconds\\"] = inference_seconds\\n    return out\\n\\n\\ndef metrics_table(rows: list[dict[str, Any]]) -> pd.DataFrame:\\n    return pd.DataFrame(rows).sort_values(\\"average_precision\\", ascending=False).reset_index(drop=True)\\n", "models.py": "\\"\\"\\"Classical and quantum model builders for the notebook PoC.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\nimport numpy as np\\nfrom sklearn.calibration import CalibratedClassifierCV\\nfrom sklearn.dummy import DummyClassifier\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.neighbors import KNeighborsClassifier\\nfrom sklearn.svm import SVC\\n\\n\\ndef make_dummy(strategy: str = \\"prior\\", seed: int = 42) -> DummyClassifier:\\n    return DummyClassifier(strategy=strategy, random_state=seed)\\n\\n\\ndef make_logistic(\\n    seed: int = 42,\\n    max_iter: int = 2000,\\n    C: float = 1.0,\\n    class_weight: str | dict | None = \\"balanced\\",\\n) -> LogisticRegression:\\n    return LogisticRegression(\\n        C=C,\\n        class_weight=class_weight,\\n        max_iter=max_iter,\\n        random_state=seed,\\n    )\\n\\n\\ndef make_linear_svm(\\n    seed: int = 42,\\n    C: float = 1.0,\\n    class_weight: str | dict | None = \\"balanced\\",\\n    calibrated: bool = True,\\n) -> Any:\\n    base = SVC(kernel=\\"linear\\", C=C, class_weight=class_weight, random_state=seed)\\n    if not calibrated:\\n        return base\\n    return CalibratedClassifierCV(base, method=\\"sigmoid\\", cv=3)\\n\\n\\ndef make_rbf_svm(\\n    seed: int = 42,\\n    C: float = 1.0,\\n    gamma: str | float = \\"scale\\",\\n    class_weight: str | dict | None = \\"balanced\\",\\n    calibrated: bool = True,\\n) -> Any:\\n    base = SVC(kernel=\\"rbf\\", C=C, gamma=gamma, class_weight=class_weight, random_state=seed)\\n    if not calibrated:\\n        return base\\n    return CalibratedClassifierCV(base, method=\\"sigmoid\\", cv=3)\\n\\n\\ndef make_precomputed_svm(\\n    seed: int = 42,\\n    C: float = 1.0,\\n    class_weight: str | dict | None = \\"balanced\\",\\n) -> SVC:\\n    return SVC(kernel=\\"precomputed\\", C=C, class_weight=class_weight, random_state=seed)\\n\\n\\ndef make_poly_svm(seed: int = 42, degree: int = 3) -> CalibratedClassifierCV:\\n    base = SVC(kernel=\\"poly\\", degree=degree, class_weight=\\"balanced\\", random_state=seed)\\n    return CalibratedClassifierCV(base, method=\\"sigmoid\\", cv=3)\\n\\n\\ndef make_knn(n_neighbors: int = 5) -> KNeighborsClassifier:\\n    return KNeighborsClassifier(n_neighbors=n_neighbors, weights=\\"distance\\")\\n\\n\\ndef make_fidelity_kernel(\\n    n_qubits: int = 4,\\n    reps: int = 2,\\n    entanglement: str = \\"linear\\",\\n    feature_map_name: str = \\"ZZFeatureMap\\",\\n    enforce_psd: bool = True,\\n) -> Any:\\n    \\"\\"\\"Build a FidelityQuantumKernel with ZZ or Z feature map.\\"\\"\\"\\n    from qiskit.circuit.library import ZFeatureMap, ZZFeatureMap\\n    from qiskit.primitives import StatevectorSampler\\n    from qiskit_machine_learning.kernels import FidelityQuantumKernel\\n\\n    name = feature_map_name.lower().replace(\\"_\\", \\"\\")\\n    if name in {\\"zfeaturemap\\", \\"z\\"}:\\n        feature_map = ZFeatureMap(feature_dimension=n_qubits, reps=reps)\\n    else:\\n        # ZZFeatureMap\\n        ent = None if entanglement in {\\"none\\", \\"None\\", None} else entanglement\\n        kwargs: dict[str, Any] = {\\n            \\"feature_dimension\\": n_qubits,\\n            \\"reps\\": reps,\\n        }\\n        if ent is not None:\\n            kwargs[\\"entanglement\\"] = ent\\n        feature_map = ZZFeatureMap(**kwargs)\\n\\n    try:\\n        from qiskit_machine_learning.state_fidelities import ComputeUncompute\\n\\n        fidelity = ComputeUncompute(sampler=StatevectorSampler())\\n        return FidelityQuantumKernel(\\n            feature_map=feature_map,\\n            fidelity=fidelity,\\n            enforce_psd=enforce_psd,\\n        )\\n    except TypeError:\\n        return FidelityQuantumKernel(feature_map=feature_map, enforce_psd=enforce_psd)\\n\\n\\ndef make_qsvc(\\n    n_qubits: int = 4,\\n    reps: int = 2,\\n    entanglement: str = \\"linear\\",\\n    feature_map_name: str = \\"ZZFeatureMap\\",\\n    enforce_psd: bool = True,\\n) -> tuple[Any, Any]:\\n    \\"\\"\\"Build FidelityQuantumKernel + QSVC. Returns (qsvc, quantum_kernel).\\"\\"\\"\\n    from qiskit_machine_learning.algorithms import QSVC\\n\\n    quantum_kernel = make_fidelity_kernel(\\n        n_qubits=n_qubits,\\n        reps=reps,\\n        entanglement=entanglement,\\n        feature_map_name=feature_map_name,\\n        enforce_psd=enforce_psd,\\n    )\\n    try:\\n        qsvc = QSVC(quantum_kernel=quantum_kernel, class_weight=\\"balanced\\")\\n    except TypeError:\\n        qsvc = QSVC(quantum_kernel=quantum_kernel)\\n    return qsvc, quantum_kernel\\n\\n\\ndef predict_proba_positive(model: Any, X: np.ndarray) -> np.ndarray | None:\\n    \\"\\"\\"Return P(class=1) if available, else a ranked score from decision_function.\\"\\"\\"\\n    if hasattr(model, \\"predict_proba\\"):\\n        proba = model.predict_proba(X)\\n        if proba.ndim == 2 and proba.shape[1] >= 2:\\n            if hasattr(model, \\"classes_\\"):\\n                classes = list(model.classes_)\\n                if 1 in classes:\\n                    return proba[:, classes.index(1)]\\n            return proba[:, -1]\\n    if hasattr(model, \\"decision_function\\"):\\n        scores = np.asarray(model.decision_function(X), dtype=float)\\n        return 1.0 / (1.0 + np.exp(-scores))\\n    return None\\n", "preprocess.py": "\\"\\"\\"Train-only imputer, scaler and PCA.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom pathlib import Path\\n\\nimport joblib\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.decomposition import PCA\\nfrom sklearn.impute import SimpleImputer\\nfrom sklearn.pipeline import Pipeline\\nfrom sklearn.preprocessing import StandardScaler\\n\\n\\ndef build_pca_pipeline(n_components: int = 4) -> Pipeline:\\n    return Pipeline(\\n        steps=[\\n            (\\"imputer\\", SimpleImputer(strategy=\\"median\\")),\\n            (\\"scaler\\", StandardScaler()),\\n            (\\"pca\\", PCA(n_components=n_components, random_state=42)),\\n        ]\\n    )\\n\\n\\ndef fit_transform_pca(\\n    train: pd.DataFrame,\\n    validation: pd.DataFrame,\\n    test: pd.DataFrame,\\n    feature_cols: list[str],\\n    n_components: int = 4,\\n) -> tuple[np.ndarray, np.ndarray, np.ndarray, Pipeline, np.ndarray]:\\n    \\"\\"\\"Fit pipeline on train features only; transform all partitions.\\"\\"\\"\\n    pipe = build_pca_pipeline(n_components=n_components)\\n    X_train = pipe.fit_transform(train[feature_cols])\\n    X_val = pipe.transform(validation[feature_cols])\\n    X_test = pipe.transform(test[feature_cols])\\n    explained = pipe.named_steps[\\"pca\\"].explained_variance_ratio_\\n    return X_train, X_val, X_test, pipe, explained\\n\\n\\ndef save_pipeline(pipe: Pipeline, path: str | Path) -> None:\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    joblib.dump(pipe, path)\\n\\n\\ndef load_pipeline(path: str | Path) -> Pipeline:\\n    return joblib.load(path)\\n", "preprocessing/__init__.py": "\\"\\"\\"Preprocessing subpackage.\\"\\"\\"\\n\\nfrom qml_air_quality.preprocessing.angular_scaling import (\\n    AngularScaler,\\n    MinMaxAngularScaler,\\n    NoneAngularScaler,\\n    QuantileAngularScaler,\\n    load_angular_scaler,\\n    make_angular_scaler,\\n    save_angular_scaler,\\n)\\n\\n__all__ = [\\n    \\"AngularScaler\\",\\n    \\"MinMaxAngularScaler\\",\\n    \\"NoneAngularScaler\\",\\n    \\"QuantileAngularScaler\\",\\n    \\"load_angular_scaler\\",\\n    \\"make_angular_scaler\\",\\n    \\"save_angular_scaler\\",\\n]\\n", "preprocessing/angular_scaling.py": "\\"\\"\\"Train-only angular scalers for quantum feature maps.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom abc import ABC, abstractmethod\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport joblib\\nimport numpy as np\\nfrom sklearn.base import BaseEstimator, TransformerMixin\\nfrom sklearn.preprocessing import MinMaxScaler, QuantileTransformer\\n\\n\\nclass AngularScaler(ABC, BaseEstimator, TransformerMixin):\\n    \\"\\"\\"Common interface: fit on train only, transform partitions.\\"\\"\\"\\n\\n    name: str = \\"base\\"\\n\\n    @abstractmethod\\n    def fit(self, X: np.ndarray, y: Any = None) -> AngularScaler:\\n        ...\\n\\n    @abstractmethod\\n    def transform(self, X: np.ndarray) -> np.ndarray:\\n        ...\\n\\n    def fit_transform(self, X: np.ndarray, y: Any = None) -> np.ndarray:\\n        return self.fit(X, y).transform(X)\\n\\n\\nclass NoneAngularScaler(AngularScaler):\\n    \\"\\"\\"Identity — control matching the original PoC (no angular remap).\\"\\"\\"\\n\\n    name = \\"none\\"\\n\\n    def fit(self, X: np.ndarray, y: Any = None) -> NoneAngularScaler:\\n        self.n_features_in_ = X.shape[1]\\n        return self\\n\\n    def transform(self, X: np.ndarray) -> np.ndarray:\\n        return np.asarray(X, dtype=float)\\n\\n\\nclass MinMaxAngularScaler(AngularScaler):\\n    \\"\\"\\"MinMax to a chosen angular range, fit on train only.\\"\\"\\"\\n\\n    def __init__(self, minimum: float = 0.0, maximum: float = np.pi, name: str | None = None):\\n        self.minimum = float(minimum)\\n        self.maximum = float(maximum)\\n        self.name = name or f\\"minmax_{minimum}_{maximum}\\"\\n        self._scaler = MinMaxScaler(feature_range=(self.minimum, self.maximum))\\n\\n    def fit(self, X: np.ndarray, y: Any = None) -> MinMaxAngularScaler:\\n        self._scaler.fit(X)\\n        self.n_features_in_ = X.shape[1]\\n        return self\\n\\n    def transform(self, X: np.ndarray) -> np.ndarray:\\n        return self._scaler.transform(X)\\n\\n\\nclass QuantileAngularScaler(AngularScaler):\\n    \\"\\"\\"QuantileTransformer to uniform [0,1], then map to [0, π].\\"\\"\\"\\n\\n    name = \\"quantile_0_pi\\"\\n\\n    def __init__(self, n_quantiles: int = 100, seed: int = 42):\\n        self.n_quantiles = n_quantiles\\n        self.seed = seed\\n        self._qt = QuantileTransformer(\\n            n_quantiles=n_quantiles,\\n            output_distribution=\\"uniform\\",\\n            random_state=seed,\\n            subsample=int(1e9),\\n        )\\n\\n    def fit(self, X: np.ndarray, y: Any = None) -> QuantileAngularScaler:\\n        n = max(10, min(self.n_quantiles, X.shape[0]))\\n        self._qt.set_params(n_quantiles=n)\\n        self._qt.fit(X)\\n        self.n_features_in_ = X.shape[1]\\n        return self\\n\\n    def transform(self, X: np.ndarray) -> np.ndarray:\\n        u = self._qt.transform(X)\\n        return np.pi * u\\n\\n\\ndef make_angular_scaler(name: str, seed: int = 42, **kwargs: Any) -> AngularScaler:\\n    \\"\\"\\"Factory for named angular strategies from the ablation plan.\\"\\"\\"\\n    key = name.lower().strip()\\n    if key in {\\"none\\", \\"identity\\", \\"sem_escala\\"}:\\n        return NoneAngularScaler()\\n    if key in {\\"minmax_0_1\\", \\"minmax_[0,1]\\", \\"[0,1]\\"}:\\n        return MinMaxAngularScaler(0.0, 1.0, name=\\"minmax_0_1\\")\\n    if key in {\\"minmax_0_pi\\", \\"minmax_[0,pi]\\", \\"[0,pi]\\", \\"minmax_0_π\\"}:\\n        return MinMaxAngularScaler(0.0, np.pi, name=\\"minmax_0_pi\\")\\n    if key in {\\n        \\"minmax_minus_pi_pi\\",\\n        \\"minmax_[-pi,pi]\\",\\n        \\"[-pi,pi]\\",\\n        \\"minmax_minus_π_π\\",\\n    }:\\n        return MinMaxAngularScaler(-np.pi, np.pi, name=\\"minmax_minus_pi_pi\\")\\n    if key in {\\"quantile_0_pi\\", \\"quantile_[0,pi]\\", \\"quantile\\"}:\\n        return QuantileAngularScaler(\\n            n_quantiles=int(kwargs.get(\\"n_quantiles\\", 100)),\\n            seed=seed,\\n        )\\n    # allow explicit min/max from yaml\\n    if \\"minimum\\" in kwargs and \\"maximum\\" in kwargs:\\n        return MinMaxAngularScaler(\\n            float(kwargs[\\"minimum\\"]),\\n            float(kwargs[\\"maximum\\"]),\\n            name=name,\\n        )\\n    raise ValueError(f\\"Unknown angular scaler: {name!r}\\")\\n\\n\\ndef save_angular_scaler(scaler: AngularScaler, path: str | Path) -> None:\\n    path = Path(path)\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    joblib.dump(scaler, path)\\n\\n\\ndef load_angular_scaler(path: str | Path) -> AngularScaler:\\n    return joblib.load(path)\\n", "evaluation/__init__.py": "\\"\\"\\"Evaluation helpers.\\"\\"\\"\\n\\nfrom qml_air_quality.evaluation.block_bootstrap import block_bootstrap_deltas\\nfrom qml_air_quality.evaluation.kernel_diagnostics import (\\n    class_contrast,\\n    diagnose_kernel,\\n    effective_rank,\\n    kernel_target_alignment,\\n    off_diagonal_stats,\\n)\\nfrom qml_air_quality.evaluation.statistical_comparison import classify_fair_result\\nfrom qml_air_quality.evaluation.threshold_selection import select_threshold_fbeta\\n\\n__all__ = [\\n    \\"block_bootstrap_deltas\\",\\n    \\"class_contrast\\",\\n    \\"classify_fair_result\\",\\n    \\"diagnose_kernel\\",\\n    \\"effective_rank\\",\\n    \\"kernel_target_alignment\\",\\n    \\"off_diagonal_stats\\",\\n    \\"select_threshold_fbeta\\",\\n]\\n", "evaluation/threshold_selection.py": "\\"\\"\\"Select decision threshold on validation by F-beta (default F2).\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\nimport numpy as np\\nfrom sklearn.metrics import fbeta_score\\n\\n\\ndef select_threshold_fbeta(\\n    y_true: np.ndarray,\\n    scores: np.ndarray,\\n    beta: float = 2.0,\\n) -> tuple[float, float, dict[str, Any]]:\\n    \\"\\"\\"Pick threshold maximizing F_beta on validation scores.\\n\\n    Returns (best_threshold, best_fbeta, metadata).\\n    \\"\\"\\"\\n    y_true = np.asarray(y_true)\\n    scores = np.asarray(scores, dtype=float)\\n    thresholds = np.unique(scores)\\n    if len(thresholds) == 0:\\n        raise ValueError(\\"no scores available for threshold selection\\")\\n\\n    best_threshold = float(thresholds[0])\\n    best_f = -1.0\\n    for thr in thresholds:\\n        preds = (scores >= thr).astype(int)\\n        f = float(fbeta_score(y_true, preds, beta=beta, pos_label=1, zero_division=0))\\n        if f > best_f:\\n            best_f = f\\n            best_threshold = float(thr)\\n\\n    meta = {\\n        \\"metric\\": f\\"f{beta:g}\\",\\n        \\"select_on\\": \\"validation\\",\\n        \\"best_threshold\\": best_threshold,\\n        \\"best_fbeta\\": best_f,\\n        \\"n_thresholds_evaluated\\": len(thresholds),\\n        \\"selection_split\\": \\"validation\\",\\n    }\\n    return best_threshold, best_f, meta\\n\\n\\ndef apply_threshold(scores: np.ndarray, threshold: float) -> np.ndarray:\\n    return (np.asarray(scores) >= threshold).astype(int)\\n", "evaluation/block_bootstrap.py": "\\"\\"\\"Temporal block bootstrap for paired model comparisons.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections.abc import Callable\\nfrom typing import Any\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.metrics import average_precision_score, fbeta_score, recall_score\\n\\n\\ndef _block_starts(n: int, block_size: int, rng: np.random.Generator) -> np.ndarray:\\n    \\"\\"\\"Sample contiguous blocks with replacement covering ~n observations.\\"\\"\\"\\n    if block_size < 1:\\n        raise ValueError(\\"block_size must be >= 1\\")\\n    n_blocks = int(np.ceil(n / block_size))\\n    max_start = max(0, n - block_size)\\n    starts = rng.integers(0, max_start + 1, size=n_blocks)\\n    idx = np.concatenate([np.arange(s, min(s + block_size, n)) for s in starts])\\n    return idx[:n]\\n\\n\\ndef block_bootstrap_deltas(\\n    y_true: np.ndarray,\\n    scores_a: np.ndarray,\\n    scores_b: np.ndarray,\\n    pred_a: np.ndarray | None = None,\\n    pred_b: np.ndarray | None = None,\\n    timestamps: np.ndarray | pd.Series | None = None,\\n    n_boot: int = 1000,\\n    block_size_hours: int = 24,\\n    seed: int = 42,\\n    confidence: float = 0.95,\\n) -> dict[str, Any]:\\n    \\"\\"\\"Block-bootstrap differences (A − B) for ranking and operational metrics.\\n\\n    Assumes hourly rows so block_size_hours maps 1:1 to consecutive indices\\n    when timestamps are sorted (caller should pass chronologically ordered data).\\n    \\"\\"\\"\\n    y_true = np.asarray(y_true)\\n    scores_a = np.asarray(scores_a, dtype=float)\\n    scores_b = np.asarray(scores_b, dtype=float)\\n    n = len(y_true)\\n    if pred_a is None:\\n        pred_a = (scores_a >= 0.5).astype(int)\\n    if pred_b is None:\\n        pred_b = (scores_b >= 0.5).astype(int)\\n    pred_a = np.asarray(pred_a)\\n    pred_b = np.asarray(pred_b)\\n\\n    rng = np.random.default_rng(seed)\\n    deltas: dict[str, list[float]] = {\\n        \\"delta_auprc\\": [],\\n        \\"delta_f2\\": [],\\n        \\"delta_recall\\": [],\\n        \\"delta_false_alert_rate\\": [],\\n    }\\n\\n    for _ in range(n_boot):\\n        idx = _block_starts(n, block_size_hours, rng)\\n        y = y_true[idx]\\n        if len(np.unique(y)) < 2:\\n            continue\\n        sa, sb = scores_a[idx], scores_b[idx]\\n        pa, pb = pred_a[idx], pred_b[idx]\\n        auprc_a = average_precision_score(y, sa)\\n        auprc_b = average_precision_score(y, sb)\\n        deltas[\\"delta_auprc\\"].append(float(auprc_a - auprc_b))\\n        f2_a = fbeta_score(y, pa, beta=2, zero_division=0)\\n        f2_b = fbeta_score(y, pb, beta=2, zero_division=0)\\n        deltas[\\"delta_f2\\"].append(float(f2_a - f2_b))\\n        rec_a = recall_score(y, pa, pos_label=1, zero_division=0)\\n        rec_b = recall_score(y, pb, pos_label=1, zero_division=0)\\n        deltas[\\"delta_recall\\"].append(float(rec_a - rec_b))\\n        far_a = float(((pa == 1) & (y == 0)).sum() / len(y))\\n        far_b = float(((pb == 1) & (y == 0)).sum() / len(y))\\n        deltas[\\"delta_false_alert_rate\\"].append(far_a - far_b)\\n\\n    alpha = (1 - confidence) / 2\\n    out: dict[str, Any] = {\\n        \\"n_boot\\": n_boot,\\n        \\"block_size_hours\\": block_size_hours,\\n        \\"n_boot_effective\\": len(deltas[\\"delta_auprc\\"]),\\n        \\"confidence\\": confidence,\\n    }\\n    for key, values in deltas.items():\\n        arr = np.asarray(values, dtype=float)\\n        out[f\\"{key}_mean\\"] = float(np.mean(arr)) if len(arr) else float(\\"nan\\")\\n        out[f\\"{key}_std\\"] = float(np.std(arr)) if len(arr) else float(\\"nan\\")\\n        out[f\\"{key}_ci_low\\"] = float(np.quantile(arr, alpha)) if len(arr) else float(\\"nan\\")\\n        out[f\\"{key}_ci_high\\"] = float(np.quantile(arr, 1 - alpha)) if len(arr) else float(\\"nan\\")\\n    return out\\n\\n\\ndef point_metric_delta(\\n    y_true: np.ndarray,\\n    scores_a: np.ndarray,\\n    scores_b: np.ndarray,\\n    pred_a: np.ndarray,\\n    pred_b: np.ndarray,\\n    metric_fn: Callable[..., float] | None = None,\\n) -> float:\\n    \\"\\"\\"Convenience for a single A−B AUPRC delta (no bootstrap).\\"\\"\\"\\n    del metric_fn\\n    return float(\\n        average_precision_score(y_true, scores_a) - average_precision_score(y_true, scores_b)\\n    )\\n", "evaluation/statistical_comparison.py": "\\"\\"\\"Classify pairwise QSVM vs classical results without overclaiming.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\n\\ndef classify_fair_result(\\n    delta_auprc_mean: float,\\n    delta_auprc_ci_low: float,\\n    delta_f2_mean: float,\\n    wins_majority_seeds: bool,\\n    beats_full_without_threshold_fix: bool = False,\\n    min_delta_auprc: float = 0.02,\\n) -> dict[str, str]:\\n    \\"\\"\\"Assign interpretation labels for the fair benchmark.\\n\\n    Never use \'quantum advantage\' wording when the CI includes zero.\\n    \\"\\"\\"\\n    clear_gain = (\\n        delta_auprc_mean >= min_delta_auprc\\n        and delta_auprc_ci_low > 0\\n        and delta_f2_mean >= 0\\n        and wins_majority_seeds\\n    )\\n    if clear_gain:\\n        return {\\n            \\"label\\": \\"CANDIDATE_QUANTUM_GAIN\\",\\n            \\"conclusion\\": (\\n                \\"QSVM superou o clássico pareado em AUPRC com IC da diferença acima de zero, \\"\\n                \\"F2 não inferior e maioria das sementes positivas. Resultado preliminar — \\"\\n                \\"não interpretar como vantagem computacional quântica geral.\\"\\n            ),\\n        }\\n    if beats_full_without_threshold_fix and not clear_gain:\\n        return {\\n            \\"label\\": \\"THRESHOLD_OR_CAPACITY_EFFECT\\",\\n            \\"conclusion\\": (\\n                \\"O ganho aparente frente ao modelo completo sem correção de limiar \\"\\n                \\"provavelmente reflete capacidade ou limiar, não o kernel quântico.\\"\\n            ),\\n        }\\n    return {\\n        \\"label\\": \\"NO_CLEAR_QUANTUM_GAIN\\",\\n        \\"conclusion\\": (\\n            \\"Empate ou diferença não estável frente ao melhor modelo pareado \\"\\n            \\"(intervalo de confiança da ΔAUPRC inclui zero ou critérios de ganho não atendidos).\\"\\n        ),\\n    }\\n\\n\\ndef majority_seed_wins(deltas_by_seed: list[float]) -> bool:\\n    if not deltas_by_seed:\\n        return False\\n    return sum(1 for d in deltas_by_seed if d > 0) > len(deltas_by_seed) / 2\\n\\n\\ndef summarize_pairwise(\\n    comparison_name: str,\\n    boot: dict[str, Any],\\n    seed_deltas: list[float],\\n    beats_full_without_threshold_fix: bool = False,\\n) -> dict[str, Any]:\\n    wins = majority_seed_wins(seed_deltas)\\n    classification = classify_fair_result(\\n        delta_auprc_mean=float(boot.get(\\"delta_auprc_mean\\", float(\\"nan\\"))),\\n        delta_auprc_ci_low=float(boot.get(\\"delta_auprc_ci_low\\", float(\\"nan\\"))),\\n        delta_f2_mean=float(boot.get(\\"delta_f2_mean\\", float(\\"nan\\"))),\\n        wins_majority_seeds=wins,\\n        beats_full_without_threshold_fix=beats_full_without_threshold_fix,\\n    )\\n    return {\\n        \\"comparison\\": comparison_name,\\n        **boot,\\n        \\"seed_delta_auprc_mean\\": float(sum(seed_deltas) / len(seed_deltas)) if seed_deltas else float(\\"nan\\"),\\n        \\"wins_majority_seeds\\": wins,\\n        **classification,\\n    }\\n", "evaluation/kernel_diagnostics.py": "\\"\\"\\"Structural diagnostics for quantum (and classical) kernel matrices.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any\\n\\nimport numpy as np\\n\\n\\ndef _off_diagonal(K: np.ndarray) -> np.ndarray:\\n    mask = ~np.eye(K.shape[0], dtype=bool)\\n    return K[mask]\\n\\n\\ndef off_diagonal_stats(K: np.ndarray) -> dict[str, float]:\\n    off = _off_diagonal(K)\\n    if off.size == 0:\\n        return {\\n            \\"off_mean\\": float(\\"nan\\"),\\n            \\"off_std\\": float(\\"nan\\"),\\n            \\"off_min\\": float(\\"nan\\"),\\n            \\"off_max\\": float(\\"nan\\"),\\n            \\"off_p05\\": float(\\"nan\\"),\\n            \\"off_p25\\": float(\\"nan\\"),\\n            \\"off_p50\\": float(\\"nan\\"),\\n            \\"off_p75\\": float(\\"nan\\"),\\n            \\"off_p95\\": float(\\"nan\\"),\\n            \\"cv_k\\": float(\\"nan\\"),\\n        }\\n    mean = float(np.mean(off))\\n    std = float(np.std(off))\\n    return {\\n        \\"off_mean\\": mean,\\n        \\"off_std\\": std,\\n        \\"off_min\\": float(np.min(off)),\\n        \\"off_max\\": float(np.max(off)),\\n        \\"off_p05\\": float(np.percentile(off, 5)),\\n        \\"off_p25\\": float(np.percentile(off, 25)),\\n        \\"off_p50\\": float(np.percentile(off, 50)),\\n        \\"off_p75\\": float(np.percentile(off, 75)),\\n        \\"off_p95\\": float(np.percentile(off, 95)),\\n        \\"cv_k\\": float(std / mean) if abs(mean) > 1e-12 else float(\\"nan\\"),\\n    }\\n\\n\\ndef class_contrast(K: np.ndarray, y: np.ndarray) -> dict[str, float]:\\n    \\"\\"\\"ΔK = μ_same − μ_different on off-diagonal pairs.\\"\\"\\"\\n    y = np.asarray(y).ravel()\\n    n = len(y)\\n    same_vals: list[float] = []\\n    diff_vals: list[float] = []\\n    for i in range(n):\\n        for j in range(i + 1, n):\\n            v = float(K[i, j])\\n            if y[i] == y[j]:\\n                same_vals.append(v)\\n            else:\\n                diff_vals.append(v)\\n    mu_same = float(np.mean(same_vals)) if same_vals else float(\\"nan\\")\\n    mu_diff = float(np.mean(diff_vals)) if diff_vals else float(\\"nan\\")\\n    return {\\n        \\"mu_same\\": mu_same,\\n        \\"mu_different\\": mu_diff,\\n        \\"delta_k\\": mu_same - mu_diff,\\n    }\\n\\n\\ndef effective_rank(K: np.ndarray, eps: float = 1e-12) -> dict[str, float]:\\n    \\"\\"\\"Shannon effective rank of eigenvalue spectrum of symmetric K.\\"\\"\\"\\n    # Numerical symmetrization\\n    Ks = 0.5 * (K + K.T)\\n    eigvals = np.linalg.eigvalsh(Ks)\\n    eigvals = np.clip(eigvals, 0.0, None)\\n    total = float(eigvals.sum())\\n    if total <= eps:\\n        return {\\"effective_rank\\": 0.0, \\"n_positive_eigs\\": 0}\\n    p = eigvals / total\\n    p = p[p > eps]\\n    entropy = float(-np.sum(p * np.log(p)))\\n    return {\\n        \\"effective_rank\\": float(np.exp(entropy)),\\n        \\"n_positive_eigs\\": int((eigvals > eps).sum()),\\n        \\"eig_max\\": float(eigvals.max()),\\n        \\"eig_min_nonneg\\": float(eigvals[eigvals > eps].min()) if (eigvals > eps).any() else 0.0,\\n    }\\n\\n\\ndef kernel_target_alignment(K: np.ndarray, y: np.ndarray) -> float:\\n    \\"\\"\\"Frobenius alignment A(K, yy^T) with labels mapped to {-1, +1}.\\"\\"\\"\\n    y = np.asarray(y).ravel().astype(float)\\n    # map {0,1} or other binary → {-1,+1}\\n    classes = np.unique(y)\\n    if len(classes) != 2:\\n        # degenerate\\n        return float(\\"nan\\")\\n    y_pm = np.where(y == classes.max(), 1.0, -1.0)\\n    Y = np.outer(y_pm, y_pm)\\n    Ks = 0.5 * (K + K.T)\\n    num = float(np.sum(Ks * Y))\\n    den = float(np.linalg.norm(Ks, ord=\\"fro\\") * np.linalg.norm(Y, ord=\\"fro\\"))\\n    if den < 1e-12:\\n        return float(\\"nan\\")\\n    return num / den\\n\\n\\ndef diagnose_kernel(K: np.ndarray, y: np.ndarray | None = None) -> dict[str, Any]:\\n    \\"\\"\\"Aggregate diagnostics for a square kernel matrix.\\"\\"\\"\\n    K = np.asarray(K, dtype=float)\\n    out: dict[str, Any] = {\\n        \\"shape\\": list(K.shape),\\n        \\"symmetric\\": bool(np.allclose(K, K.T, atol=1e-6)),\\n        \\"diag_mean\\": float(np.mean(np.diag(K))),\\n        \\"finite\\": bool(np.all(np.isfinite(K))),\\n    }\\n    out.update(off_diagonal_stats(K))\\n    out.update(effective_rank(K))\\n    if y is not None:\\n        out.update(class_contrast(K, y))\\n        out[\\"alignment\\"] = kernel_target_alignment(K, y)\\n    return out\\n", "experiments/__init__.py": "\\"\\"\\"Experiment runners.\\"\\"\\"\\n\\nfrom qml_air_quality.experiments.fair_benchmark import run_fair_benchmark\\nfrom qml_air_quality.experiments.final_evaluation import run_final_evaluation\\nfrom qml_air_quality.experiments.quantum_ablation import run_ablation\\n\\n__all__ = [\\"run_ablation\\", \\"run_fair_benchmark\\", \\"run_final_evaluation\\"]\\n", "experiments/fair_benchmark.py": "\\"\\"\\"Farooq-style fair benchmark: purge, fixed eval sets, paired PCA-2, two stages.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport logging\\nimport time\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.metrics import average_precision_score\\n\\nfrom qml_air_quality.config import load_config, project_root\\nfrom qml_air_quality.data import download_dataset, load_station_csv\\nfrom qml_air_quality.evaluation.block_bootstrap import block_bootstrap_deltas\\nfrom qml_air_quality.evaluation.statistical_comparison import summarize_pairwise\\nfrom qml_air_quality.evaluation.threshold_selection import apply_threshold\\nfrom qml_air_quality.experiments.model_selection import (\\n    freeze_selection,\\n    ranking_scores,\\n    select_logistic,\\n    select_qsvm,\\n    select_svm,\\n)\\nfrom qml_air_quality.features import (\\n    add_farooq_style_stats,\\n    add_future_target,\\n    apply_causal_ffill,\\n    impute_with_train_medians,\\n    make_binary_target,\\n    save_target_metadata,\\n)\\nfrom qml_air_quality.metrics import binary_metrics\\nfrom qml_air_quality.models import (\\n    make_fidelity_kernel,\\n    make_linear_svm,\\n    make_logistic,\\n    make_precomputed_svm,\\n    make_rbf_svm,\\n)\\nfrom qml_air_quality.paired_representation import (\\n    fit_paired_representation,\\n    save_preprocessors,\\n)\\nfrom qml_air_quality.reporting.fair_benchmark_plots import generate_fair_benchmark_plots\\nfrom qml_air_quality.sample_registry import (\\n    SampleRegistry,\\n    build_fixed_evaluation_sets,\\n    sample_train_for_seed,\\n)\\nfrom qml_air_quality.split import apply_temporal_purge, temporal_split\\n\\nlogger = logging.getLogger(__name__)\\n\\nQUANTUM_CONFIGS = [\\n    {\\"id\\": \\"Q01\\", \\"angular_scaler\\": \\"minmax_0_1\\", \\"reps\\": 1, \\"farooq_style\\": False},\\n    {\\"id\\": \\"Q02\\", \\"angular_scaler\\": \\"minmax_0_1\\", \\"reps\\": 2, \\"farooq_style\\": True},\\n    {\\"id\\": \\"Q03\\", \\"angular_scaler\\": \\"minmax_0_pi\\", \\"reps\\": 1, \\"farooq_style\\": False},\\n    {\\"id\\": \\"Q04\\", \\"angular_scaler\\": \\"minmax_0_pi\\", \\"reps\\": 2, \\"farooq_style\\": False},\\n]\\n\\nANGULAR_ALIASES = {\\n    \\"minmax_0_1\\": \\"0_1\\",\\n    \\"minmax_0_pi\\": \\"0_pi\\",\\n}\\n\\n\\ndef _artifact_dirs(cfg: dict[str, Any]) -> dict[str, Path]:\\n    root = project_root()\\n    base = root / cfg[\\"paths\\"][\\"artifacts_dir\\"]\\n    dirs = {\\n        \\"base\\": base,\\n        \\"samples\\": base / \\"sample_indices\\",\\n        \\"preprocessors\\": base / \\"preprocessors\\",\\n        \\"kernels\\": base / \\"kernels\\",\\n        \\"predictions\\": base / \\"predictions\\",\\n        \\"embeddings\\": base / \\"embeddings\\",\\n    }\\n    for d in dirs.values():\\n        d.mkdir(parents=True, exist_ok=True)\\n    return dirs\\n\\n\\ndef _persist_prediction(\\n    directory: Path,\\n    key: str,\\n    *,\\n    scores: np.ndarray,\\n    preds: np.ndarray,\\n    y: np.ndarray,\\n    timestamps: np.ndarray,\\n    original_index: np.ndarray | None = None,\\n    threshold: float,\\n    model_id: str,\\n    seed: int,\\n    train_size: int,\\n) -> None:\\n    \\"\\"\\"Save raw scores/preds as npz + csv for every model×seed×n.\\"\\"\\"\\n    safe = key.replace(\\"|\\", \\"_\\")\\n    payload: dict[str, Any] = {\\n        \\"scores\\": np.asarray(scores, dtype=float),\\n        \\"preds\\": np.asarray(preds, dtype=int),\\n        \\"y\\": np.asarray(y, dtype=int),\\n        \\"timestamps\\": np.asarray(timestamps),\\n        \\"threshold\\": np.asarray(threshold),\\n    }\\n    if original_index is not None:\\n        payload[\\"original_index\\"] = np.asarray(original_index)\\n    np.savez(directory / f\\"{safe}.npz\\", **payload)\\n\\n    rows = {\\n        \\"timestamp\\": pd.to_datetime(timestamps),\\n        \\"y_true\\": y,\\n        \\"score\\": scores,\\n        \\"y_pred\\": preds,\\n        \\"threshold\\": threshold,\\n        \\"model\\": model_id,\\n        \\"seed\\": seed,\\n        \\"train_size\\": train_size,\\n    }\\n    if original_index is not None:\\n        rows[\\"original_index\\"] = original_index\\n    pd.DataFrame(rows).to_csv(directory / f\\"{safe}.csv\\", index=False)\\n\\n\\ndef prepare_fair_frames(cfg: dict[str, Any]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, list[str], dict, dict]:\\n    \\"\\"\\"Download → Farooq stats → split → purge → P90 on purged train.\\"\\"\\"\\n    root = project_root()\\n    csv_path = download_dataset(\\n        dataset_id=cfg[\\"data\\"][\\"dataset_id\\"],\\n        station=cfg[\\"data\\"][\\"station\\"],\\n        raw_dir=root / cfg[\\"data\\"][\\"raw_dir\\"],\\n    )\\n    df = load_station_csv(csv_path)\\n    source_map = dict(cfg[\\"features\\"][\\"source_columns\\"])\\n    base_cols = list(source_map.keys())\\n    df = apply_causal_ffill(df, base_cols, limit_hours=cfg[\\"features\\"][\\"ffill_limit_hours\\"])\\n    df, feat_cols = add_farooq_style_stats(\\n        df,\\n        source_map=source_map,\\n        window=int(cfg[\\"features\\"][\\"rolling_window_hours\\"]),\\n        min_periods=int(cfg[\\"features\\"][\\"rolling_min_periods\\"]),\\n    )\\n    df = add_future_target(\\n        df,\\n        target_column=cfg[\\"data\\"][\\"target_column\\"],\\n        horizon_hours=cfg[\\"experiment\\"][\\"horizon_hours\\"],\\n    )\\n    df = df.dropna(subset=[\\"future_pm25\\"] + feat_cols).reset_index(drop=True)\\n\\n    train, val, test = temporal_split(\\n        df,\\n        train_fraction=cfg[\\"split\\"][\\"train_fraction\\"],\\n        validation_fraction=cfg[\\"split\\"][\\"validation_fraction\\"],\\n        test_fraction=cfg[\\"split\\"][\\"test_fraction\\"],\\n    )\\n    purge_hours = int(cfg[\\"data\\"].get(\\"purge_hours\\", cfg[\\"experiment\\"][\\"horizon_hours\\"]))\\n    train, val, test, purge_meta = apply_temporal_purge(train, val, test, purge_hours=purge_hours)\\n\\n    train, val, test = impute_with_train_medians(train, val, test, columns=feat_cols)\\n    train, val, test, tmeta = make_binary_target(\\n        train,\\n        val,\\n        test,\\n        percentile=cfg[\\"experiment\\"][\\"extreme_percentile\\"],\\n        source=\\"purged_training_partition\\",\\n        horizon_hours=cfg[\\"experiment\\"][\\"horizon_hours\\"],\\n    )\\n    audit = {\\n        \\"station\\": cfg[\\"data\\"][\\"station\\"],\\n        \\"n_rows_after_dropna\\": len(df),\\n        \\"feature_columns\\": feat_cols,\\n        \\"train_size_full\\": len(train),\\n        \\"validation_size_full\\": len(val),\\n        \\"test_size_full\\": len(test),\\n        \\"train_positive_rate\\": float(train[\\"target\\"].mean()),\\n    }\\n    return train, val, test, feat_cols, tmeta, {**purge_meta, **audit}\\n\\n\\ndef _cw(value: Any) -> str | None:\\n    if value is None or value == \\"null\\":\\n        return None\\n    return value\\n\\n\\ndef _classical_grids(cfg: dict[str, Any]) -> dict[str, Any]:\\n    return {\\n        \\"logistic_C\\": cfg[\\"classical\\"][\\"logistic\\"][\\"C\\"],\\n        \\"logistic_cw\\": cfg[\\"classical\\"][\\"logistic\\"][\\"class_weight\\"],\\n        \\"svm_linear_C\\": cfg[\\"classical\\"][\\"svm_linear\\"][\\"C\\"],\\n        \\"svm_linear_cw\\": cfg[\\"classical\\"][\\"svm_linear\\"][\\"class_weight\\"],\\n        \\"svm_rbf_C\\": cfg[\\"classical\\"][\\"svm_rbf\\"][\\"C\\"],\\n        \\"svm_rbf_cw\\": cfg[\\"classical\\"][\\"svm_rbf\\"][\\"class_weight\\"],\\n        \\"svm_rbf_gamma\\": cfg[\\"classical\\"][\\"svm_rbf\\"][\\"gamma\\"],\\n        \\"qsvm_C\\": cfg[\\"quantum\\"][\\"C\\"],\\n        \\"qsvm_cw\\": cfg[\\"quantum\\"][\\"class_weight\\"],\\n    }\\n\\n\\ndef _eval_row(\\n    name: str,\\n    family: str,\\n    y_true: np.ndarray,\\n    scores: np.ndarray,\\n    threshold: float,\\n    *,\\n    seed: int,\\n    train_size: int,\\n    extra: dict[str, Any] | None = None,\\n) -> dict[str, Any]:\\n    preds = apply_threshold(scores, threshold)\\n    row = binary_metrics(y_true, preds, scores, model_name=name)\\n    row.update(\\n        {\\n            \\"family\\": family,\\n            \\"seed\\": seed,\\n            \\"train_size\\": train_size,\\n            \\"threshold\\": threshold,\\n            \\"split\\": \\"test\\",\\n        }\\n    )\\n    if extra:\\n        row.update(extra)\\n    return row\\n\\n\\ndef _fit_frozen_classical(\\n    family: str,\\n    params: dict[str, Any],\\n    X_train: np.ndarray,\\n    y_train: np.ndarray,\\n    seed: int,\\n) -> Any:\\n    if family == \\"logistic\\":\\n        return make_logistic(\\n            seed=seed,\\n            C=float(params[\\"C\\"]),\\n            class_weight=_cw(params.get(\\"class_weight\\")),\\n        ).fit(X_train, y_train)\\n    if family == \\"svm_linear\\":\\n        return make_linear_svm(\\n            seed=seed,\\n            C=float(params[\\"C\\"]),\\n            class_weight=_cw(params.get(\\"class_weight\\")),\\n            calibrated=False,\\n        ).fit(X_train, y_train)\\n    if family == \\"svm_rbf\\":\\n        gamma = params.get(\\"gamma\\", \\"scale\\")\\n        return make_rbf_svm(\\n            seed=seed,\\n            C=float(params[\\"C\\"]),\\n            gamma=gamma,\\n            class_weight=_cw(params.get(\\"class_weight\\")),\\n            calibrated=False,\\n        ).fit(X_train, y_train)\\n    raise ValueError(family)\\n\\n\\ndef run_selection_stage(\\n    cfg: dict[str, Any],\\n    train_full: pd.DataFrame,\\n    val_fixed: pd.DataFrame,\\n    feat_cols: list[str],\\n    paths: dict[str, Path],\\n    *,\\n    skip_quantum: bool = False,\\n) -> pd.DataFrame:\\n    \\"\\"\\"Stage A: hyperparameter + quantum config selection on validation; test blocked.\\"\\"\\"\\n    grids = _classical_grids(cfg)\\n    seeds = list(cfg[\\"experiment\\"][\\"selection_seeds\\"])\\n    train_size = int(cfg[\\"sampling\\"][\\"train_sizes\\"][0])\\n    registry = SampleRegistry(paths[\\"samples\\"])\\n    rows: list[dict[str, Any]] = []\\n    kernel_cache: dict[str, np.ndarray] = {}\\n\\n    for seed in seeds:\\n        logger.info(\\"selection seed=%s train_size=%s\\", seed, train_size)\\n        tr = sample_train_for_seed(train_full, train_size=train_size, seed=seed)\\n        registry.save_train(tr, seed=seed, train_size=train_size)\\n        y_tr = tr[\\"target\\"].to_numpy()\\n        y_va = val_fixed[\\"target\\"].to_numpy()\\n\\n        repr_ = fit_paired_representation(tr, val_fixed, val_fixed, feat_cols, pca_components=2, seed=seed)\\n        save_preprocessors(repr_, paths[\\"preprocessors\\"] / f\\"seed_{seed}_n{train_size}\\")\\n\\n        # Group A — full 8D\\n        full_selections: list[tuple[str, dict[str, Any]]] = [\\n            (\\n                \\"logistic_full_8d\\",\\n                select_logistic(\\n                    repr_.X_train_full,\\n                    y_tr,\\n                    repr_.X_val_full,\\n                    y_va,\\n                    grids[\\"logistic_C\\"],\\n                    grids[\\"logistic_cw\\"],\\n                    seed=seed,\\n                ),\\n            ),\\n            (\\n                \\"svm_linear_full_8d\\",\\n                select_svm(\\n                    repr_.X_train_full,\\n                    y_tr,\\n                    repr_.X_val_full,\\n                    y_va,\\n                    kernel=\\"linear\\",\\n                    C_grid=grids[\\"svm_linear_C\\"],\\n                    class_weight_grid=grids[\\"svm_linear_cw\\"],\\n                    seed=seed,\\n                ),\\n            ),\\n            (\\n                \\"svm_rbf_full_8d\\",\\n                select_svm(\\n                    repr_.X_train_full,\\n                    y_tr,\\n                    repr_.X_val_full,\\n                    y_va,\\n                    kernel=\\"rbf\\",\\n                    C_grid=grids[\\"svm_rbf_C\\"],\\n                    class_weight_grid=grids[\\"svm_rbf_cw\\"],\\n                    gamma_grid=grids[\\"svm_rbf_gamma\\"],\\n                    seed=seed,\\n                ),\\n            ),\\n        ]\\n        for name, sel in full_selections:\\n            frozen = freeze_selection(sel)\\n            frozen.update(\\n                {\\n                    \\"model_id\\": name,\\n                    \\"group\\": \\"full\\",\\n                    \\"seed\\": seed,\\n                    \\"train_size\\": train_size,\\n                    \\"angular_scaler\\": None,\\n                    \\"reps\\": None,\\n                }\\n            )\\n            rows.append(frozen)\\n\\n        # Group B — PCA2 + angular scales\\n        for ang_name, ang_tag in ANGULAR_ALIASES.items():\\n            X_tr_a, X_va_a, _, _, _ = repr_.angular(ang_name, seed=seed)\\n            paired_selections: list[tuple[str, dict[str, Any]]] = [\\n                (\\n                    f\\"logistic_pca2_{ang_tag}\\",\\n                    select_logistic(\\n                        X_tr_a,\\n                        y_tr,\\n                        X_va_a,\\n                        y_va,\\n                        grids[\\"logistic_C\\"],\\n                        grids[\\"logistic_cw\\"],\\n                        seed=seed,\\n                    ),\\n                ),\\n                (\\n                    f\\"svm_linear_pca2_{ang_tag}\\",\\n                    select_svm(\\n                        X_tr_a,\\n                        y_tr,\\n                        X_va_a,\\n                        y_va,\\n                        kernel=\\"linear\\",\\n                        C_grid=grids[\\"svm_linear_C\\"],\\n                        class_weight_grid=grids[\\"svm_linear_cw\\"],\\n                        seed=seed,\\n                    ),\\n                ),\\n                (\\n                    f\\"svm_rbf_pca2_{ang_tag}\\",\\n                    select_svm(\\n                        X_tr_a,\\n                        y_tr,\\n                        X_va_a,\\n                        y_va,\\n                        kernel=\\"rbf\\",\\n                        C_grid=grids[\\"svm_rbf_C\\"],\\n                        class_weight_grid=grids[\\"svm_rbf_cw\\"],\\n                        gamma_grid=grids[\\"svm_rbf_gamma\\"],\\n                        seed=seed,\\n                    ),\\n                ),\\n            ]\\n            for name, sel in paired_selections:\\n                frozen = freeze_selection(sel)\\n                frozen.update(\\n                    {\\n                        \\"model_id\\": name,\\n                        \\"group\\": \\"paired\\",\\n                        \\"seed\\": seed,\\n                        \\"train_size\\": train_size,\\n                        \\"angular_scaler\\": ang_name,\\n                        \\"reps\\": None,\\n                    }\\n                )\\n                rows.append(frozen)\\n\\n            if skip_quantum:\\n                continue\\n\\n            for qcfg in QUANTUM_CONFIGS:\\n                if qcfg[\\"angular_scaler\\"] != ang_name:\\n                    continue\\n                qid = qcfg[\\"id\\"]\\n                cache_key = f\\"{qid}_seed{seed}_n{train_size}\\"\\n                t0 = time.perf_counter()\\n                sel = select_qsvm(\\n                    X_tr_a,\\n                    y_tr,\\n                    X_va_a,\\n                    y_va,\\n                    C_grid=grids[\\"qsvm_C\\"],\\n                    class_weight_grid=grids[\\"qsvm_cw\\"],\\n                    reps=int(qcfg[\\"reps\\"]),\\n                    feature_map=cfg[\\"quantum\\"][\\"feature_map\\"],\\n                    entanglement=cfg[\\"quantum\\"][\\"entanglement\\"],\\n                    seed=seed,\\n                    kernel_cache=kernel_cache,\\n                    cache_key=cache_key,\\n                )\\n                kernel_s = time.perf_counter() - t0\\n                np.save(paths[\\"kernels\\"] / f\\"{cache_key}_K_train.npy\\", sel[\\"K_train\\"])\\n                np.save(paths[\\"kernels\\"] / f\\"{cache_key}_K_val.npy\\", sel[\\"K_val\\"])\\n                frozen = freeze_selection(sel)\\n                frozen.update(\\n                    {\\n                        \\"model_id\\": f\\"qsvm_pca2_{ang_tag}_reps{qcfg[\'reps\']}\\",\\n                        \\"quantum_id\\": qid,\\n                        \\"farooq_style\\": qcfg[\\"farooq_style\\"],\\n                        \\"group\\": \\"paired\\",\\n                        \\"seed\\": seed,\\n                        \\"train_size\\": train_size,\\n                        \\"angular_scaler\\": ang_name,\\n                        \\"reps\\": qcfg[\\"reps\\"],\\n                        \\"kernel_seconds\\": kernel_s,\\n                    }\\n                )\\n                rows.append(frozen)\\n\\n    selection_df = pd.DataFrame(rows)\\n    selection_df.to_csv(paths[\\"base\\"] / \\"model_selection.csv\\", index=False)\\n\\n    # Aggregate: pick best hyperparams per model_id by mean val_auprc across seeds\\n    def _mode_or_first(s: pd.Series) -> Any:\\n        s = s.dropna()\\n        if s.empty:\\n            return None\\n        modes = s.mode()\\n        return modes.iloc[0] if len(modes) else s.iloc[0]\\n\\n    agg_spec: dict[str, Any] = {\\n        \\"val_auprc\\": (\\"val_auprc\\", \\"mean\\"),\\n        \\"val_recall_extreme\\": (\\"val_recall_extreme\\", \\"mean\\"),\\n        \\"C\\": (\\"C\\", _mode_or_first),\\n        \\"class_weight\\": (\\"class_weight\\", _mode_or_first),\\n        \\"gamma\\": (\\"gamma\\", _mode_or_first),\\n        \\"angular_scaler\\": (\\"angular_scaler\\", \\"first\\"),\\n        \\"reps\\": (\\"reps\\", \\"first\\"),\\n        \\"group\\": (\\"group\\", \\"first\\"),\\n        \\"threshold\\": (\\"threshold\\", \\"median\\"),\\n    }\\n    if \\"quantum_id\\" in selection_df.columns:\\n        agg_spec[\\"quantum_id\\"] = (\\"quantum_id\\", \\"first\\")\\n    if \\"farooq_style\\" in selection_df.columns:\\n        agg_spec[\\"farooq_style\\"] = (\\"farooq_style\\", \\"first\\")\\n    agg = selection_df.groupby(\\"model_id\\", as_index=False).agg(**agg_spec)\\n    # Prefer quantum config by mean val AUPRC among QSVM models\\n    q_mask = agg[\\"model_id\\"].astype(str).str.startswith(\\"qsvm_\\")\\n    models_records = json.loads(agg.to_json(orient=\\"records\\"))\\n    if q_mask.any():\\n        best_q = agg.loc[q_mask].sort_values(\\n            [\\"val_auprc\\", \\"val_recall_extreme\\"], ascending=False\\n        ).iloc[0]\\n        frozen_cfg = {\\n            \\"selection_split\\": \\"validation\\",\\n            \\"best_quantum_model_id\\": best_q[\\"model_id\\"],\\n            \\"best_quantum_id\\": None if pd.isna(best_q.get(\\"quantum_id\\")) else best_q.get(\\"quantum_id\\"),\\n            \\"best_quantum_angular_scaler\\": best_q[\\"angular_scaler\\"],\\n            \\"best_quantum_reps\\": int(best_q[\\"reps\\"]) if pd.notna(best_q[\\"reps\\"]) else None,\\n            \\"models\\": models_records,\\n        }\\n    else:\\n        frozen_cfg = {\\"selection_split\\": \\"validation\\", \\"models\\": models_records}\\n\\n    (paths[\\"base\\"] / \\"selection_frozen.json\\").write_text(\\n        json.dumps(frozen_cfg, indent=2, allow_nan=False), encoding=\\"utf-8\\"\\n    )\\n    return selection_df\\n\\n\\ndef _params_for_model(frozen: dict[str, Any], model_id: str) -> dict[str, Any]:\\n    for m in frozen[\\"models\\"]:\\n        if m[\\"model_id\\"] == model_id:\\n            return m\\n    raise KeyError(model_id)\\n\\n\\ndef run_final_stage(\\n    cfg: dict[str, Any],\\n    train_full: pd.DataFrame,\\n    val_fixed: pd.DataFrame,\\n    test_fixed: pd.DataFrame,\\n    feat_cols: list[str],\\n    paths: dict[str, Path],\\n    *,\\n    skip_quantum: bool = False,\\n) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:\\n    \\"\\"\\"Stage B: frozen configs, 10 seeds, train sizes 200/500, test unlocked.\\"\\"\\"\\n    frozen_path = paths[\\"base\\"] / \\"selection_frozen.json\\"\\n    frozen = json.loads(frozen_path.read_text(encoding=\\"utf-8\\"))\\n    seeds = list(cfg[\\"experiment\\"][\\"final_seeds\\"])\\n    train_sizes = list(cfg[\\"sampling\\"][\\"train_sizes\\"])\\n    registry = SampleRegistry(paths[\\"samples\\"])\\n    metric_rows: list[dict[str, Any]] = []\\n    pred_store: dict[str, dict[str, Any]] = {}\\n\\n    # Models to evaluate in final: full refs + paired for selected angular + best Q\\n    best_ang = frozen.get(\\"best_quantum_angular_scaler\\") or \\"minmax_0_1\\"\\n    best_reps = frozen.get(\\"best_quantum_reps\\") or 2\\n    ang_tag = ANGULAR_ALIASES[best_ang]\\n    final_model_ids = [\\n        \\"logistic_full_8d\\",\\n        \\"svm_linear_full_8d\\",\\n        \\"svm_rbf_full_8d\\",\\n        f\\"logistic_pca2_{ang_tag}\\",\\n        f\\"svm_linear_pca2_{ang_tag}\\",\\n        f\\"svm_rbf_pca2_{ang_tag}\\",\\n    ]\\n    q_model_id = f\\"qsvm_pca2_{ang_tag}_reps{best_reps}\\"\\n    if not skip_quantum:\\n        final_model_ids.append(q_model_id)\\n\\n    for train_size in train_sizes:\\n        for seed in seeds:\\n            logger.info(\\"final seed=%s train_size=%s\\", seed, train_size)\\n            tr = sample_train_for_seed(train_full, train_size=train_size, seed=seed)\\n            registry.save_train(tr, seed=seed, train_size=train_size)\\n            y_tr = tr[\\"target\\"].to_numpy()\\n            y_va = val_fixed[\\"target\\"].to_numpy()\\n            y_te = test_fixed[\\"target\\"].to_numpy()\\n            ts_te = test_fixed[\\"timestamp\\"].to_numpy()\\n            idx_te = (\\n                test_fixed[\\"original_index\\"].to_numpy()\\n                if \\"original_index\\" in test_fixed.columns\\n                else None\\n            )\\n\\n            repr_ = fit_paired_representation(\\n                tr, val_fixed, test_fixed, feat_cols, pca_components=2, seed=seed\\n            )\\n            X_tr_a, X_va_a, X_te_a, _, _ = repr_.angular(best_ang, seed=seed)\\n\\n            # Raw embeddings for this seed / train size (paired space = quantum inputs)\\n            np.savez(\\n                paths[\\"embeddings\\"] / f\\"seed{seed}_n{train_size}.npz\\",\\n                X_train_full=repr_.X_train_full,\\n                X_val_full=repr_.X_val_full,\\n                X_test_full=repr_.X_test_full,\\n                X_train_pca=repr_.X_train_pca,\\n                X_val_pca=repr_.X_val_pca,\\n                X_test_pca=repr_.X_test_pca,\\n                X_train_angular=X_tr_a,\\n                X_val_angular=X_va_a,\\n                X_test_angular=X_te_a,\\n                y_train=y_tr,\\n                y_val=y_va,\\n                y_test=y_te,\\n                angular_scaler=np.asarray(best_ang),\\n                feature_cols=np.asarray(feat_cols),\\n            )\\n\\n            for mid in final_model_ids:\\n                params = _params_for_model(frozen, mid)\\n                thr = float(params[\\"threshold\\"])\\n\\n                if mid.endswith(\\"_full_8d\\"):\\n                    family = mid.replace(\\"_full_8d\\", \\"\\")\\n                    model = _fit_frozen_classical(family, params, repr_.X_train_full, y_tr, seed)\\n                    scores = ranking_scores(model, repr_.X_test_full)\\n                    preds = apply_threshold(scores, thr)\\n                    row = _eval_row(\\n                        mid, \\"classical_full\\", y_te, scores, thr,\\n                        seed=seed, train_size=train_size,\\n                        extra={\\"angular_scaler\\": None, \\"reps\\": None},\\n                    )\\n                    metric_rows.append(row)\\n                    key = f\\"{mid}|{seed}|{train_size}\\"\\n                    pred_store[key] = {\\n                        \\"scores\\": scores,\\n                        \\"preds\\": preds,\\n                        \\"y\\": y_te,\\n                        \\"timestamps\\": ts_te,\\n                    }\\n                    _persist_prediction(\\n                        paths[\\"predictions\\"],\\n                        key,\\n                        scores=scores,\\n                        preds=preds,\\n                        y=y_te,\\n                        timestamps=ts_te,\\n                        original_index=idx_te,\\n                        threshold=thr,\\n                        model_id=mid,\\n                        seed=seed,\\n                        train_size=train_size,\\n                    )\\n                    continue\\n\\n                if mid.startswith(\\"qsvm_\\"):\\n                    qk = make_fidelity_kernel(\\n                        n_qubits=X_tr_a.shape[1],\\n                        reps=int(best_reps),\\n                        entanglement=cfg[\\"quantum\\"][\\"entanglement\\"],\\n                        feature_map_name=cfg[\\"quantum\\"][\\"feature_map\\"],\\n                    )\\n                    t0 = time.perf_counter()\\n                    K_tr = qk.evaluate(x_vec=X_tr_a)\\n                    K_te = qk.evaluate(x_vec=X_te_a, y_vec=X_tr_a)\\n                    kernel_s = time.perf_counter() - t0\\n                    np.save(\\n                        paths[\\"kernels\\"] / f\\"final_{mid}_seed{seed}_n{train_size}_K_train.npy\\",\\n                        K_tr,\\n                    )\\n                    np.save(\\n                        paths[\\"kernels\\"] / f\\"final_{mid}_seed{seed}_n{train_size}_K_test.npy\\",\\n                        K_te,\\n                    )\\n                    model = make_precomputed_svm(\\n                        seed=seed,\\n                        C=float(params[\\"C\\"]),\\n                        class_weight=_cw(params.get(\\"class_weight\\")),\\n                    )\\n                    model.fit(K_tr, y_tr)\\n                    scores = ranking_scores(model, K_te)\\n                    preds = apply_threshold(scores, thr)\\n                    row = _eval_row(\\n                        mid, \\"qsvm\\", y_te, scores, thr,\\n                        seed=seed, train_size=train_size,\\n                        extra={\\n                            \\"angular_scaler\\": best_ang,\\n                            \\"reps\\": best_reps,\\n                            \\"kernel_seconds\\": kernel_s,\\n                            \\"farooq_style\\": best_ang == \\"minmax_0_1\\" and best_reps == 2,\\n                        },\\n                    )\\n                    metric_rows.append(row)\\n                    key = f\\"{mid}|{seed}|{train_size}\\"\\n                    pred_store[key] = {\\n                        \\"scores\\": scores,\\n                        \\"preds\\": preds,\\n                        \\"y\\": y_te,\\n                        \\"timestamps\\": ts_te,\\n                    }\\n                    _persist_prediction(\\n                        paths[\\"predictions\\"],\\n                        key,\\n                        scores=scores,\\n                        preds=preds,\\n                        y=y_te,\\n                        timestamps=ts_te,\\n                        original_index=idx_te,\\n                        threshold=thr,\\n                        model_id=mid,\\n                        seed=seed,\\n                        train_size=train_size,\\n                    )\\n                    continue\\n\\n                # paired classical on angular PCA space\\n                family = mid.split(\\"_pca2_\\")[0]\\n                model = _fit_frozen_classical(family, params, X_tr_a, y_tr, seed)\\n                scores = ranking_scores(model, X_te_a)\\n                preds = apply_threshold(scores, thr)\\n                row = _eval_row(\\n                    mid, \\"classical_paired\\", y_te, scores, thr,\\n                    seed=seed, train_size=train_size,\\n                    extra={\\"angular_scaler\\": best_ang, \\"reps\\": None},\\n                )\\n                metric_rows.append(row)\\n                key = f\\"{mid}|{seed}|{train_size}\\"\\n                pred_store[key] = {\\n                    \\"scores\\": scores,\\n                    \\"preds\\": preds,\\n                    \\"y\\": y_te,\\n                    \\"timestamps\\": ts_te,\\n                }\\n                _persist_prediction(\\n                    paths[\\"predictions\\"],\\n                    key,\\n                    scores=scores,\\n                    preds=preds,\\n                    y=y_te,\\n                    timestamps=ts_te,\\n                    original_index=idx_te,\\n                    threshold=thr,\\n                    model_id=mid,\\n                    seed=seed,\\n                    train_size=train_size,\\n                )\\n\\n            del y_va, X_va_a\\n\\n    metrics_df = pd.DataFrame(metric_rows)\\n    metrics_df.to_csv(paths[\\"base\\"] / \\"final_metrics_by_seed.csv\\", index=False)\\n\\n    summary = (\\n        metrics_df.groupby([\\"model\\", \\"train_size\\", \\"family\\"], as_index=False)\\n        .agg(\\n            average_precision_mean=(\\"average_precision\\", \\"mean\\"),\\n            average_precision_std=(\\"average_precision\\", \\"std\\"),\\n            auroc_mean=(\\"auroc\\", \\"mean\\"),\\n            balanced_accuracy_mean=(\\"balanced_accuracy\\", \\"mean\\"),\\n            precision_mean=(\\"precision_extreme\\", \\"mean\\"),\\n            recall_mean=(\\"recall_extreme\\", \\"mean\\"),\\n            f1_mean=(\\"f1_extreme\\", \\"mean\\"),\\n            f2_mean=(\\"f2_extreme\\", \\"mean\\"),\\n            mcc_mean=(\\"mcc\\", \\"mean\\"),\\n            false_alert_rate_mean=(\\"false_alert_rate\\", \\"mean\\"),\\n            missed_extreme_rate_mean=(\\"missed_extreme_rate\\", \\"mean\\"),\\n            n_seeds=(\\"seed\\", \\"nunique\\"),\\n        )\\n    )\\n    summary.to_csv(paths[\\"base\\"] / \\"final_metrics_summary.csv\\", index=False)\\n\\n    # Block bootstrap pairwise on pooled last train_size or both\\n    boot_rows: list[dict[str, Any]] = []\\n    report: dict[str, Any] = {\\"selection\\": frozen, \\"classifications\\": []}\\n\\n    for train_size in train_sizes:\\n        # Pool predictions across seeds for a fixed train_size\\n        def _pool(\\n            model_id: str, n_train: int\\n        ) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:\\n            ys, scores, preds, tss = [], [], [], []\\n            for seed in seeds:\\n                key = f\\"{model_id}|{seed}|{n_train}\\"\\n                if key not in pred_store:\\n                    continue\\n                p = pred_store[key]\\n                ys.append(p[\\"y\\"])\\n                scores.append(p[\\"scores\\"])\\n                preds.append(p[\\"preds\\"])\\n                tss.append(p[\\"timestamps\\"])\\n            if not ys:\\n                raise KeyError(model_id)\\n            return (\\n                np.concatenate(ys),\\n                np.concatenate(scores),\\n                np.concatenate(preds),\\n                np.concatenate(tss),\\n            )\\n\\n        q_id = q_model_id if not skip_quantum else None\\n        if q_id and any(k.startswith(f\\"{q_id}|\\") and k.endswith(f\\"|{train_size}\\") for k in pred_store):\\n            y_q, s_q, p_q, ts = _pool(q_id, train_size)\\n            comparisons = [\\n                (f\\"svm_rbf_pca2_{ang_tag}\\", \\"QSVM − SVM-RBF paired\\"),\\n                (f\\"logistic_pca2_{ang_tag}\\", \\"QSVM − logistic paired\\"),\\n                (\\"svm_rbf_full_8d\\", \\"QSVM − best classic full proxy (svm_rbf_full)\\"),\\n            ]\\n            for other_id, label in comparisons:\\n                if not any(k.startswith(f\\"{other_id}|\\") and k.endswith(f\\"|{train_size}\\") for k in pred_store):\\n                    continue\\n                _, s_o, p_o, _ = _pool(other_id, train_size)\\n                # Align lengths (same test set per seed so concat order matches)\\n                boot = block_bootstrap_deltas(\\n                    y_q,\\n                    s_q,\\n                    s_o,\\n                    pred_a=p_q,\\n                    pred_b=p_o,\\n                    timestamps=ts,\\n                    n_boot=int(cfg[\\"evaluation\\"][\\"bootstrap_iterations\\"]),\\n                    block_size_hours=int(cfg[\\"evaluation\\"][\\"bootstrap_block_hours\\"]),\\n                    seed=42,\\n                )\\n                seed_deltas = []\\n                for seed in seeds:\\n                    kq = f\\"{q_id}|{seed}|{train_size}\\"\\n                    ko = f\\"{other_id}|{seed}|{train_size}\\"\\n                    if kq in pred_store and ko in pred_store:\\n                        seed_deltas.append(\\n                            float(\\n                                average_precision_score(pred_store[kq][\\"y\\"], pred_store[kq][\\"scores\\"])\\n                                - average_precision_score(pred_store[ko][\\"y\\"], pred_store[ko][\\"scores\\"])\\n                            )\\n                        )\\n                summary_row = summarize_pairwise(\\n                    f\\"{label} [n={train_size}]\\",\\n                    boot,\\n                    seed_deltas,\\n                    beats_full_without_threshold_fix=False,\\n                )\\n                summary_row[\\"train_size\\"] = train_size\\n                summary_row[\\"model_a\\"] = q_id\\n                summary_row[\\"model_b\\"] = other_id\\n                boot_rows.append(summary_row)\\n                report[\\"classifications\\"].append(summary_row)\\n\\n    boot_df = pd.DataFrame(boot_rows)\\n    boot_df.to_csv(paths[\\"base\\"] / \\"pairwise_bootstrap.csv\\", index=False)\\n\\n    return metrics_df, boot_df, report\\n\\n\\ndef write_final_report(\\n    paths: dict[str, Path],\\n    tmeta: dict[str, Any],\\n    split_meta: dict[str, Any],\\n    report: dict[str, Any],\\n    summary_df: pd.DataFrame,\\n) -> None:\\n    lines = [\\n        \\"# Farooq-style fair benchmark — relatório final\\",\\n        \\"\\",\\n        \\"## Alvo\\",\\n        f\\"- percentil: {tmeta.get(\'percentile\')}\\",\\n        f\\"- limiar: {tmeta.get(\'threshold\')}\\",\\n        f\\"- fonte: {tmeta.get(\'source\')}\\",\\n        \\"\\",\\n        \\"## Split e purga\\",\\n        f\\"- train rows após purga: {split_meta.get(\'train_rows_after\')}\\",\\n        f\\"- validation rows após purga: {split_meta.get(\'validation_rows_after\')}\\",\\n        f\\"- test rows: {split_meta.get(\'test_rows\')}\\",\\n        \\"\\",\\n        \\"## Resumo de métricas (teste)\\",\\n        \\"\\",\\n        summary_df.to_string(index=False),\\n        \\"\\",\\n        \\"## Comparações pareadas (block bootstrap)\\",\\n        \\"\\",\\n    ]\\n    for c in report.get(\\"classifications\\", []):\\n        lines.append(\\n            f\\"- **{c.get(\'comparison\')}**: label=`{c.get(\'label\')}` \\"\\n            f\\"ΔAUPRC={c.get(\'delta_auprc_mean\'):.4f} \\"\\n            f\\"IC=[{c.get(\'delta_auprc_ci_low\'):.4f}, {c.get(\'delta_auprc_ci_high\'):.4f}]\\"\\n        )\\n        lines.append(f\\"  - {c.get(\'conclusion\')}\\")\\n    lines.append(\\"\\")\\n    lines.append(\\n        \\"Nota: não usar o termo \'vantagem quântica\' quando o intervalo de confiança inclui zero.\\"\\n    )\\n    (paths[\\"base\\"] / \\"final_report.md\\").write_text(\\"\\\\n\\".join(lines), encoding=\\"utf-8\\")\\n\\n\\ndef _deep_update(base: dict[str, Any], overrides: dict[str, Any]) -> dict[str, Any]:\\n    \\"\\"\\"Recursively merge overrides into a copy of base.\\"\\"\\"\\n    out = dict(base)\\n    for key, value in overrides.items():\\n        if isinstance(value, dict) and isinstance(out.get(key), dict):\\n            out[key] = _deep_update(out[key], value)\\n        else:\\n            out[key] = value\\n    return out\\n\\n\\ndef run_fair_benchmark(\\n    config_path: str | Path,\\n    stage: str = \\"all\\",\\n    skip_quantum: bool = False,\\n    overrides: dict[str, Any] | None = None,\\n) -> dict[str, Any]:\\n    \\"\\"\\"Entry point for selection / final / all stages.\\"\\"\\"\\n    root = project_root()\\n    cfg = load_config(root / config_path if not Path(config_path).is_absolute() else config_path)\\n    if overrides:\\n        cfg = _deep_update(cfg, overrides)\\n    paths = _artifact_dirs(cfg)\\n    (paths[\\"base\\"] / \\"run_config.json\\").write_text(\\n        json.dumps(\\n            {\\n                \\"config_path\\": str(config_path),\\n                \\"stage\\": stage,\\n                \\"skip_quantum\\": skip_quantum,\\n                \\"overrides\\": overrides or {},\\n                \\"sampling\\": cfg.get(\\"sampling\\"),\\n                \\"experiment_seeds\\": {\\n                    \\"selection\\": cfg[\\"experiment\\"][\\"selection_seeds\\"],\\n                    \\"final\\": cfg[\\"experiment\\"][\\"final_seeds\\"],\\n                },\\n                \\"evaluation\\": cfg.get(\\"evaluation\\"),\\n                \\"artifacts_dir\\": cfg[\\"paths\\"][\\"artifacts_dir\\"],\\n            },\\n            indent=2,\\n            default=str,\\n        ),\\n        encoding=\\"utf-8\\",\\n    )\\n\\n    train, val, test, feat_cols, tmeta, split_meta = prepare_fair_frames(cfg)\\n    save_target_metadata(tmeta, paths[\\"base\\"] / \\"target_metadata.json\\")\\n    (paths[\\"base\\"] / \\"split_metadata.json\\").write_text(\\n        json.dumps(split_meta, indent=2, default=str), encoding=\\"utf-8\\"\\n    )\\n    (paths[\\"base\\"] / \\"data_audit.json\\").write_text(\\n        json.dumps(\\n            {\\n                \\"station\\": cfg[\\"data\\"][\\"station\\"],\\n                \\"features\\": feat_cols,\\n                \\"n_train\\": len(train),\\n                \\"n_validation\\": len(val),\\n                \\"n_test\\": len(test),\\n                \\"threshold\\": tmeta[\\"threshold\\"],\\n            },\\n            indent=2,\\n        ),\\n        encoding=\\"utf-8\\",\\n    )\\n\\n    va_fixed, te_fixed, sample_meta = build_fixed_evaluation_sets(\\n        val,\\n        test,\\n        validation_size=int(cfg[\\"sampling\\"][\\"validation_size\\"]),\\n        test_size=int(cfg[\\"sampling\\"][\\"test_size\\"]),\\n        evaluation_seed=int(cfg[\\"sampling\\"][\\"evaluation_seed\\"]),\\n    )\\n    registry = SampleRegistry(paths[\\"samples\\"])\\n    registry.save_fixed(va_fixed, te_fixed)\\n    (paths[\\"base\\"] / \\"sample_meta.json\\").write_text(\\n        json.dumps({k: v for k, v in sample_meta.items() if k not in {\\"validation_indices\\", \\"test_indices\\"}}, indent=2),\\n        encoding=\\"utf-8\\",\\n    )\\n\\n    result: dict[str, Any] = {\\"target_metadata\\": tmeta, \\"split_metadata\\": split_meta}\\n\\n    if stage in {\\"selection\\", \\"all\\"}:\\n        sel = run_selection_stage(\\n            cfg, train, va_fixed, feat_cols, paths, skip_quantum=skip_quantum\\n        )\\n        result[\\"selection\\"] = sel\\n\\n    if stage in {\\"final\\", \\"all\\"}:\\n        if not (paths[\\"base\\"] / \\"selection_frozen.json\\").exists():\\n            raise FileNotFoundError(\\"selection_frozen.json missing; run selection stage first\\")\\n        metrics_df, boot_df, report = run_final_stage(\\n            cfg, train, va_fixed, te_fixed, feat_cols, paths, skip_quantum=skip_quantum\\n        )\\n        summary = pd.read_csv(paths[\\"base\\"] / \\"final_metrics_summary.csv\\")\\n        write_final_report(paths, tmeta, split_meta, report, summary)\\n        plot_paths = generate_fair_benchmark_plots(paths[\\"base\\"])\\n        result[\\"metrics\\"] = metrics_df\\n        result[\\"bootstrap\\"] = boot_df\\n        result[\\"report\\"] = report\\n        result[\\"plots\\"] = [str(p) for p in plot_paths]\\n\\n    return result\\n", "experiments/model_selection.py": "\\"\\"\\"Hyperparameter and quantum-config selection on validation only.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport itertools\\nfrom typing import Any\\n\\nimport numpy as np\\nfrom sklearn.linear_model import LogisticRegression\\nfrom sklearn.metrics import average_precision_score\\nfrom sklearn.svm import SVC\\n\\nfrom qml_air_quality.evaluation.threshold_selection import select_threshold_fbeta\\nfrom qml_air_quality.models import (\\n    make_fidelity_kernel,\\n    make_precomputed_svm,\\n    predict_proba_positive,\\n)\\n\\n\\ndef _resolve_class_weight(value: Any) -> str | None:\\n    if value is None or value == \\"null\\" or value == \\"None\\":\\n        return None\\n    return value\\n\\n\\ndef ranking_scores(model: Any, X: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"Continuous scores for AUPRC / threshold search.\\"\\"\\"\\n    if hasattr(model, \\"decision_function\\"):\\n        return np.asarray(model.decision_function(X), dtype=float)\\n    proba = predict_proba_positive(model, X)\\n    if proba is None:\\n        raise RuntimeError(\\"model cannot produce ranking scores\\")\\n    return np.asarray(proba, dtype=float)\\n\\n\\ndef _score_candidate(\\n    y_val: np.ndarray,\\n    scores: np.ndarray,\\n) -> tuple[float, float]:\\n    auprc = float(average_precision_score(y_val, scores))\\n    _, _, meta = select_threshold_fbeta(y_val, scores, beta=2.0)\\n    # temporary F2 after best threshold for tie-break context; recall used later\\n    thr = meta[\\"best_threshold\\"]\\n    preds = (scores >= thr).astype(int)\\n    recall = float(((preds == 1) & (y_val == 1)).sum() / max(1, int((y_val == 1).sum())))\\n    return auprc, recall\\n\\n\\ndef select_logistic(\\n    X_train: np.ndarray,\\n    y_train: np.ndarray,\\n    X_val: np.ndarray,\\n    y_val: np.ndarray,\\n    C_grid: list[float],\\n    class_weight_grid: list[Any],\\n    seed: int = 42,\\n) -> dict[str, Any]:\\n    best: dict[str, Any] | None = None\\n    for C, cw in itertools.product(C_grid, class_weight_grid):\\n        model = LogisticRegression(\\n            C=C,\\n            class_weight=_resolve_class_weight(cw),\\n            max_iter=2000,\\n            random_state=seed,\\n        )\\n        model.fit(X_train, y_train)\\n        scores = ranking_scores(model, X_val)\\n        auprc, recall = _score_candidate(y_val, scores)\\n        cand = {\\n            \\"model_family\\": \\"logistic\\",\\n            \\"C\\": C,\\n            \\"class_weight\\": cw,\\n            \\"val_auprc\\": auprc,\\n            \\"val_recall_extreme\\": recall,\\n            \\"selection_split\\": \\"validation\\",\\n        }\\n        if best is None or (auprc, recall) > (best[\\"val_auprc\\"], best[\\"val_recall_extreme\\"]):\\n            best = cand\\n            best[\\"model\\"] = model\\n            best[\\"val_scores\\"] = scores\\n    assert best is not None\\n    thr, f2, tmeta = select_threshold_fbeta(y_val, best[\\"val_scores\\"], beta=2.0)\\n    best[\\"threshold\\"] = thr\\n    best[\\"val_f2\\"] = f2\\n    best[\\"threshold_meta\\"] = tmeta\\n    return best\\n\\n\\ndef select_svm(\\n    X_train: np.ndarray,\\n    y_train: np.ndarray,\\n    X_val: np.ndarray,\\n    y_val: np.ndarray,\\n    *,\\n    kernel: str,\\n    C_grid: list[float],\\n    class_weight_grid: list[Any],\\n    gamma_grid: list[Any] | None = None,\\n    seed: int = 42,\\n) -> dict[str, Any]:\\n    gamma_grid = gamma_grid or [\\"scale\\"]\\n    best: dict[str, Any] | None = None\\n    for C, cw, gamma in itertools.product(C_grid, class_weight_grid, gamma_grid):\\n        kwargs: dict[str, Any] = {\\n            \\"kernel\\": kernel,\\n            \\"C\\": C,\\n            \\"class_weight\\": _resolve_class_weight(cw),\\n            \\"random_state\\": seed,\\n        }\\n        if kernel == \\"rbf\\":\\n            kwargs[\\"gamma\\"] = gamma\\n        model = SVC(**kwargs)\\n        model.fit(X_train, y_train)\\n        scores = ranking_scores(model, X_val)\\n        auprc, recall = _score_candidate(y_val, scores)\\n        cand = {\\n            \\"model_family\\": f\\"svm_{kernel}\\",\\n            \\"C\\": C,\\n            \\"class_weight\\": cw,\\n            \\"gamma\\": gamma if kernel == \\"rbf\\" else None,\\n            \\"val_auprc\\": auprc,\\n            \\"val_recall_extreme\\": recall,\\n            \\"selection_split\\": \\"validation\\",\\n        }\\n        if best is None or (auprc, recall) > (best[\\"val_auprc\\"], best[\\"val_recall_extreme\\"]):\\n            best = cand\\n            best[\\"model\\"] = model\\n            best[\\"val_scores\\"] = scores\\n    assert best is not None\\n    thr, f2, tmeta = select_threshold_fbeta(y_val, best[\\"val_scores\\"], beta=2.0)\\n    best[\\"threshold\\"] = thr\\n    best[\\"val_f2\\"] = f2\\n    best[\\"threshold_meta\\"] = tmeta\\n    return best\\n\\n\\ndef select_qsvm(\\n    X_train: np.ndarray,\\n    y_train: np.ndarray,\\n    X_val: np.ndarray,\\n    y_val: np.ndarray,\\n    *,\\n    C_grid: list[float],\\n    class_weight_grid: list[Any],\\n    reps: int,\\n    feature_map: str = \\"ZZFeatureMap\\",\\n    entanglement: str = \\"linear\\",\\n    seed: int = 42,\\n    kernel_cache: dict[str, np.ndarray] | None = None,\\n    cache_key: str | None = None,\\n) -> dict[str, Any]:\\n    n_qubits = X_train.shape[1]\\n    if kernel_cache is not None and cache_key and f\\"{cache_key}_train\\" in kernel_cache:\\n        K_tr = kernel_cache[f\\"{cache_key}_train\\"]\\n        K_va = kernel_cache[f\\"{cache_key}_val\\"]\\n    else:\\n        qk = make_fidelity_kernel(\\n            n_qubits=n_qubits,\\n            reps=reps,\\n            entanglement=entanglement,\\n            feature_map_name=feature_map,\\n            enforce_psd=True,\\n        )\\n        K_tr = qk.evaluate(x_vec=X_train)\\n        K_va = qk.evaluate(x_vec=X_val, y_vec=X_train)\\n        if kernel_cache is not None and cache_key:\\n            kernel_cache[f\\"{cache_key}_train\\"] = K_tr\\n            kernel_cache[f\\"{cache_key}_val\\"] = K_va\\n\\n    best: dict[str, Any] | None = None\\n    for C, cw in itertools.product(C_grid, class_weight_grid):\\n        model = make_precomputed_svm(\\n            seed=seed,\\n            C=C,\\n            class_weight=_resolve_class_weight(cw),\\n        )\\n        model.fit(K_tr, y_train)\\n        scores = ranking_scores(model, K_va)\\n        auprc, recall = _score_candidate(y_val, scores)\\n        cand = {\\n            \\"model_family\\": \\"qsvm\\",\\n            \\"C\\": C,\\n            \\"class_weight\\": cw,\\n            \\"reps\\": reps,\\n            \\"feature_map\\": feature_map,\\n            \\"entanglement\\": entanglement,\\n            \\"val_auprc\\": auprc,\\n            \\"val_recall_extreme\\": recall,\\n            \\"selection_split\\": \\"validation\\",\\n        }\\n        if best is None or (auprc, recall) > (best[\\"val_auprc\\"], best[\\"val_recall_extreme\\"]):\\n            best = cand\\n            best[\\"model\\"] = model\\n            best[\\"val_scores\\"] = scores\\n            best[\\"K_train\\"] = K_tr\\n            best[\\"K_val\\"] = K_va\\n    assert best is not None\\n    thr, f2, tmeta = select_threshold_fbeta(y_val, best[\\"val_scores\\"], beta=2.0)\\n    best[\\"threshold\\"] = thr\\n    best[\\"val_f2\\"] = f2\\n    best[\\"threshold_meta\\"] = tmeta\\n    return best\\n\\n\\ndef freeze_selection(selected: dict[str, Any]) -> dict[str, Any]:\\n    \\"\\"\\"Drop fitted objects; keep hyperparameters + threshold for final stage.\\"\\"\\"\\n    skip = {\\"model\\", \\"val_scores\\", \\"K_train\\", \\"K_val\\", \\"threshold_meta\\"}\\n    out = {k: v for k, v in selected.items() if k not in skip}\\n    out[\\"selection_split\\"] = \\"validation\\"\\n    out[\\"frozen\\"] = True\\n    return out\\n", "experiments/final_evaluation.py": "\\"\\"\\"Final evaluation stage wrapper for the fair benchmark.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nfrom qml_air_quality.experiments.fair_benchmark import run_fair_benchmark\\n\\n\\ndef run_final_evaluation(\\n    config_path: str | Path,\\n    skip_quantum: bool = False,\\n) -> dict[str, Any]:\\n    \\"\\"\\"Run only the final (frozen) evaluation stage.\\"\\"\\"\\n    return run_fair_benchmark(config_path, stage=\\"final\\", skip_quantum=skip_quantum)\\n", "experiments/quantum_ablation.py": "\\"\\"\\"Leakage-free quantum angular ablation experiment (Q01–Q10).\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport logging\\nimport time\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.svm import SVC\\n\\nfrom qml_air_quality.config import load_config, project_root\\nfrom qml_air_quality.data import download_dataset, load_station_csv\\nfrom qml_air_quality.evaluation.kernel_diagnostics import diagnose_kernel\\nfrom qml_air_quality.features import (\\n    add_calendar_features,\\n    add_future_target,\\n    add_lag_and_rolling,\\n    apply_causal_ffill,\\n    feature_columns,\\n    impute_with_train_medians,\\n    make_binary_target,\\n)\\nfrom qml_air_quality.metrics import binary_metrics\\nfrom qml_air_quality.models import (\\n    make_dummy,\\n    make_fidelity_kernel,\\n    make_knn,\\n    make_linear_svm,\\n    make_logistic,\\n    make_poly_svm,\\n    make_rbf_svm,\\n    predict_proba_positive,\\n)\\nfrom qml_air_quality.preprocess import fit_transform_pca\\nfrom qml_air_quality.preprocessing.angular_scaling import (\\n    make_angular_scaler,\\n    save_angular_scaler,\\n)\\nfrom qml_air_quality.split import stratified_subsample, temporal_split\\n\\nlogger = logging.getLogger(__name__)\\n\\n\\n@dataclass\\nclass PartitionData:\\n    X_train: np.ndarray\\n    X_val: np.ndarray\\n    X_test: np.ndarray\\n    y_train: np.ndarray\\n    y_val: np.ndarray\\n    y_test: np.ndarray\\n\\n\\ndef _ensure_dirs(cfg: dict[str, Any]) -> dict[str, Path]:\\n    root = project_root()\\n    paths = {\\n        \\"ablation\\": root / cfg[\\"paths\\"][\\"ablation_dir\\"],\\n        \\"kernels\\": root / cfg[\\"paths\\"][\\"kernels_dir\\"],\\n        \\"plots\\": root / cfg[\\"paths\\"][\\"plots_dir\\"],\\n        \\"reports\\": root / cfg[\\"paths\\"][\\"reports_dir\\"],\\n    }\\n    for p in paths.values():\\n        p.mkdir(parents=True, exist_ok=True)\\n    return paths\\n\\n\\ndef prepare_temporal_frames(poc_cfg: dict[str, Any]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, list[str], dict]:\\n    \\"\\"\\"Reproduce the leakage-free prepare pipeline; return train/val/test + features + target meta.\\"\\"\\"\\n    root = project_root()\\n    csv_path = download_dataset(\\n        dataset_id=poc_cfg[\\"data\\"][\\"dataset_id\\"],\\n        station=poc_cfg[\\"data\\"][\\"station\\"],\\n        raw_dir=root / poc_cfg[\\"data\\"][\\"raw_dir\\"],\\n    )\\n    df = load_station_csv(csv_path)\\n    raw_cols = poc_cfg[\\"features\\"][\\"raw_columns\\"]\\n    df = apply_causal_ffill(df, raw_cols, limit_hours=poc_cfg[\\"features\\"][\\"ffill_limit_hours\\"])\\n    df = add_lag_and_rolling(\\n        df,\\n        columns=raw_cols,\\n        lag_hours=poc_cfg[\\"features\\"][\\"lag_hours\\"],\\n        rolling_windows=poc_cfg[\\"features\\"][\\"rolling_windows\\"],\\n    )\\n    if poc_cfg[\\"features\\"][\\"add_calendar_features\\"]:\\n        df = add_calendar_features(df)\\n    df = add_future_target(\\n        df,\\n        target_column=poc_cfg[\\"data\\"][\\"target_column\\"],\\n        horizon_hours=poc_cfg[\\"forecast\\"][\\"horizon_hours\\"],\\n    )\\n    train, val, test = temporal_split(\\n        df,\\n        train_fraction=poc_cfg[\\"split\\"][\\"train_fraction\\"],\\n        validation_fraction=poc_cfg[\\"split\\"][\\"validation_fraction\\"],\\n        test_fraction=poc_cfg[\\"split\\"][\\"test_fraction\\"],\\n    )\\n    feat_cols = feature_columns(train)\\n    train, val, test = impute_with_train_medians(train, val, test, columns=feat_cols)\\n    train, val, test, tmeta = make_binary_target(\\n        train, val, test, percentile=poc_cfg[\\"forecast\\"][\\"extreme_percentile\\"]\\n    )\\n    return train, val, test, feat_cols, tmeta\\n\\n\\ndef build_pca_partitions(\\n    train: pd.DataFrame,\\n    val: pd.DataFrame,\\n    test: pd.DataFrame,\\n    feat_cols: list[str],\\n    n_components: int,\\n    train_size: int,\\n    val_size: int,\\n    test_size: int,\\n    seed: int,\\n) -> PartitionData:\\n    X_tr, X_va, X_te, _, _ = fit_transform_pca(\\n        train, val, test, feat_cols, n_components=n_components\\n    )\\n    tr = train.copy()\\n    va = val.copy()\\n    te = test.copy()\\n    for i in range(n_components):\\n        tr[f\\"pc{i + 1}\\"] = X_tr[:, i]\\n        va[f\\"pc{i + 1}\\"] = X_va[:, i]\\n        te[f\\"pc{i + 1}\\"] = X_te[:, i]\\n    tr_s, va_s, te_s, _ = stratified_subsample(\\n        tr, va, te, train_size=train_size, validation_size=val_size, test_size=test_size, seed=seed\\n    )\\n    pcs = [f\\"pc{i + 1}\\" for i in range(n_components)]\\n    return PartitionData(\\n        X_train=tr_s[pcs].to_numpy(),\\n        X_val=va_s[pcs].to_numpy(),\\n        X_test=te_s[pcs].to_numpy(),\\n        y_train=tr_s[\\"target\\"].to_numpy(),\\n        y_val=va_s[\\"target\\"].to_numpy(),\\n        y_test=te_s[\\"target\\"].to_numpy(),\\n    )\\n\\n\\ndef apply_angular(part: PartitionData, scaler_name: str, seed: int) -> tuple[PartitionData, Any]:\\n    scaler = make_angular_scaler(scaler_name, seed=seed)\\n    X_train = scaler.fit_transform(part.X_train)\\n    X_val = scaler.transform(part.X_val)\\n    X_test = scaler.transform(part.X_test)\\n    return (\\n        PartitionData(X_train, X_val, X_test, part.y_train, part.y_val, part.y_test),\\n        scaler,\\n    )\\n\\n\\ndef evaluate_precomputed_svm(\\n    K_train: np.ndarray,\\n    K_eval: np.ndarray,\\n    y_train: np.ndarray,\\n    y_eval: np.ndarray,\\n    seed: int,\\n    model_name: str,\\n) -> dict[str, Any]:\\n    clf = SVC(kernel=\\"precomputed\\", class_weight=\\"balanced\\", random_state=seed)\\n    t0 = time.perf_counter()\\n    clf.fit(K_train, y_train)\\n    train_s = time.perf_counter() - t0\\n    t1 = time.perf_counter()\\n    y_hat = clf.predict(K_eval)\\n    scores = clf.decision_function(K_eval)\\n    y_prob = 1.0 / (1.0 + np.exp(-scores))\\n    inf_s = time.perf_counter() - t1\\n    return binary_metrics(\\n        y_eval,\\n        y_hat,\\n        y_prob,\\n        model_name=model_name,\\n        training_seconds=train_s,\\n        inference_seconds=inf_s,\\n    )\\n\\n\\ndef evaluate_classical_on_embedding(\\n    part: PartitionData,\\n    seed: int,\\n    split: str = \\"val\\",\\n) -> list[dict[str, Any]]:\\n    X_tr, y_tr = part.X_train, part.y_train\\n    if split == \\"val\\":\\n        X_ev, y_ev = part.X_val, part.y_val\\n    else:\\n        X_ev, y_ev = part.X_test, part.y_test\\n\\n    models = {\\n        \\"dummy\\": make_dummy(seed=seed),\\n        \\"logistic\\": make_logistic(seed=seed),\\n        \\"svm_linear\\": make_linear_svm(seed=seed),\\n        \\"svm_rbf\\": make_rbf_svm(seed=seed),\\n        \\"svm_poly\\": make_poly_svm(seed=seed),\\n        \\"knn\\": make_knn(),\\n    }\\n    rows = []\\n    for name, model in models.items():\\n        t0 = time.perf_counter()\\n        model.fit(X_tr, y_tr)\\n        train_s = time.perf_counter() - t0\\n        t1 = time.perf_counter()\\n        y_hat = model.predict(X_ev)\\n        y_prob = predict_proba_positive(model, X_ev)\\n        inf_s = time.perf_counter() - t1\\n        rows.append(\\n            binary_metrics(\\n                y_ev,\\n                y_hat,\\n                y_prob,\\n                model_name=name,\\n                training_seconds=train_s,\\n                inference_seconds=inf_s,\\n            )\\n        )\\n    return rows\\n\\n\\ndef _kernel_paths(kernels_dir: Path, cfg_id: str, seed: int) -> dict[str, Path]:\\n    return {\\n        \\"train\\": kernels_dir / f\\"{cfg_id}_seed{seed}_K_train.npy\\",\\n        \\"val\\": kernels_dir / f\\"{cfg_id}_seed{seed}_K_val.npy\\",\\n        \\"test\\": kernels_dir / f\\"{cfg_id}_seed{seed}_K_test.npy\\",\\n        \\"meta\\": kernels_dir / f\\"{cfg_id}_seed{seed}_meta.json\\",\\n    }\\n\\n\\ndef compute_or_load_kernels(\\n    qkernel: Any,\\n    part: PartitionData,\\n    paths: dict[str, Path],\\n    cfg_id: str,\\n    seed: int,\\n    force: bool = False,\\n) -> tuple[np.ndarray, np.ndarray, np.ndarray, float]:\\n    if (\\n        not force\\n        and paths[\\"train\\"].exists()\\n        and paths[\\"val\\"].exists()\\n        and paths[\\"test\\"].exists()\\n    ):\\n        logger.info(\\"Loading cached kernels for %s\\", cfg_id)\\n        return (\\n            np.load(paths[\\"train\\"]),\\n            np.load(paths[\\"val\\"]),\\n            np.load(paths[\\"test\\"]),\\n            float(json.loads(paths[\\"meta\\"].read_text()).get(\\"kernel_seconds\\", float(\\"nan\\"))),\\n        )\\n\\n    t0 = time.perf_counter()\\n    logger.info(\\"Evaluating quantum kernel for %s …\\", cfg_id)\\n    K_train = qkernel.evaluate(x_vec=part.X_train)\\n    K_val = qkernel.evaluate(x_vec=part.X_val, y_vec=part.X_train)\\n    K_test = qkernel.evaluate(x_vec=part.X_test, y_vec=part.X_train)\\n    kernel_seconds = time.perf_counter() - t0\\n    np.save(paths[\\"train\\"], K_train)\\n    np.save(paths[\\"val\\"], K_val)\\n    np.save(paths[\\"test\\"], K_test)\\n    paths[\\"meta\\"].write_text(\\n        json.dumps({\\"id\\": cfg_id, \\"seed\\": seed, \\"kernel_seconds\\": kernel_seconds}, indent=2),\\n        encoding=\\"utf-8\\",\\n    )\\n    return K_train, K_val, K_test, kernel_seconds\\n\\n\\ndef rank_key(row: dict[str, Any]) -> tuple:\\n    \\"\\"\\"Selection order: val AUPRC, recall, alignment, lower kernel time.\\"\\"\\"\\n    return (\\n        -float(row.get(\\"average_precision\\", 0.0)),\\n        -float(row.get(\\"recall_extreme\\", 0.0)),\\n        -float(row.get(\\"alignment\\", 0.0) or 0.0),\\n        float(row.get(\\"kernel_seconds\\", 1e9) or 1e9),\\n    )\\n\\n\\ndef run_ablation(\\n    ablation_config_path: str | Path | None = None,\\n    force_kernels: bool = False,\\n    skip_test: bool = False,\\n) -> dict[str, Any]:\\n    \\"\\"\\"Run full Q01–Q10 validation ranking; optionally final test of selected models.\\"\\"\\"\\n    root = project_root()\\n    abl_cfg = load_config(ablation_config_path or (root / \\"config\\" / \\"quantum_ablation.yaml\\"))\\n    poc_cfg = load_config(root / abl_cfg[\\"data\\"].get(\\"config_ref\\", \\"config/poc.yaml\\"))\\n    paths = _ensure_dirs(abl_cfg)\\n    seed = int(abl_cfg[\\"project\\"][\\"seeds\\"][0])\\n    np.random.seed(seed)\\n\\n    logger.info(\\"Preparing temporal dataset…\\")\\n    train, val, test, feat_cols, tmeta = prepare_temporal_frames(poc_cfg)\\n    (paths[\\"ablation\\"] / \\"target_metadata.json\\").write_text(json.dumps(tmeta, indent=2), encoding=\\"utf-8\\")\\n\\n    sampling = abl_cfg[\\"sampling\\"]\\n    pca_cache: dict[int, PartitionData] = {}\\n    for n in abl_cfg[\\"dimensionality\\"][\\"pca_components\\"]:\\n        pca_cache[int(n)] = build_pca_partitions(\\n            train,\\n            val,\\n            test,\\n            feat_cols,\\n            n_components=int(n),\\n            train_size=sampling[\\"train_size\\"],\\n            val_size=sampling[\\"validation_size\\"],\\n            test_size=sampling[\\"test_size\\"],\\n            seed=seed,\\n        )\\n\\n    val_rows: list[dict[str, Any]] = []\\n    classical_val_rows: list[dict[str, Any]] = []\\n\\n    # Paired classical on each PCA dim (no angular) — for selection baseline\\n    for n, part in pca_cache.items():\\n        for row in evaluate_classical_on_embedding(part, seed=seed, split=\\"val\\"):\\n            row = {**row, \\"pca_components\\": n, \\"angular_scaler\\": \\"none\\", \\"split\\": \\"val\\", \\"family\\": \\"classical_paired\\"}\\n            classical_val_rows.append(row)\\n\\n    for qcfg in abl_cfg[\\"quantum_configs\\"]:\\n        cfg_id = qcfg[\\"id\\"]\\n        n = int(qcfg[\\"pca_components\\"])\\n        part0 = pca_cache[n]\\n        part, scaler = apply_angular(part0, qcfg[\\"angular_scaler\\"], seed=seed)\\n        save_angular_scaler(scaler, paths[\\"ablation\\"] / f\\"{cfg_id}_angular_scaler.joblib\\")\\n\\n        qkernel = make_fidelity_kernel(\\n            n_qubits=n,\\n            reps=int(qcfg[\\"reps\\"]),\\n            entanglement=qcfg.get(\\"entanglement\\", \\"linear\\"),\\n            feature_map_name=qcfg[\\"feature_map\\"],\\n        )\\n        kpaths = _kernel_paths(paths[\\"kernels\\"], cfg_id, seed)\\n        K_train, K_val, K_test, kernel_seconds = compute_or_load_kernels(\\n            qkernel, part, kpaths, cfg_id, seed, force=force_kernels\\n        )\\n        diag = diagnose_kernel(K_train, part.y_train)\\n        metrics = evaluate_precomputed_svm(\\n            K_train, K_val, part.y_train, part.y_val, seed=seed, model_name=f\\"qsvm_{cfg_id}\\"\\n        )\\n        row = {\\n            **metrics,\\n            \\"id\\": cfg_id,\\n            \\"pca_components\\": n,\\n            \\"angular_scaler\\": qcfg[\\"angular_scaler\\"],\\n            \\"feature_map\\": qcfg[\\"feature_map\\"],\\n            \\"reps\\": qcfg[\\"reps\\"],\\n            \\"entanglement\\": qcfg.get(\\"entanglement\\"),\\n            \\"kernel_seconds\\": kernel_seconds,\\n            \\"alignment\\": diag.get(\\"alignment\\"),\\n            \\"effective_rank\\": diag.get(\\"effective_rank\\"),\\n            \\"delta_k\\": diag.get(\\"delta_k\\"),\\n            \\"cv_k\\": diag.get(\\"cv_k\\"),\\n            \\"off_mean\\": diag.get(\\"off_mean\\"),\\n            \\"off_std\\": diag.get(\\"off_std\\"),\\n            \\"split\\": \\"val\\",\\n            \\"family\\": \\"qsvm\\",\\n            \\"diagnostics\\": diag,\\n        }\\n        val_rows.append(row)\\n        logger.info(\\n            \\"%s val AUPRC=%.4f align=%.4f ΔK=%.4f\\",\\n            cfg_id,\\n            row[\\"average_precision\\"],\\n            row.get(\\"alignment\\") or 0.0,\\n            row.get(\\"delta_k\\") or 0.0,\\n        )\\n\\n    val_df = pd.DataFrame([{k: v for k, v in r.items() if k != \\"diagnostics\\"} for r in val_rows])\\n    val_df = val_df.sort_values(by=[\\"average_precision\\", \\"recall_extreme\\", \\"alignment\\"], ascending=False)\\n    val_df.to_csv(paths[\\"ablation\\"] / \\"validation_ranking.csv\\", index=False)\\n    pd.DataFrame(classical_val_rows).to_csv(paths[\\"ablation\\"] / \\"classical_paired_validation.csv\\", index=False)\\n\\n    # Persist full diagnostics\\n    diag_path = paths[\\"ablation\\"] / \\"kernel_diagnostics_val.json\\"\\n    diag_path.write_text(\\n        json.dumps({r[\\"id\\"]: r[\\"diagnostics\\"] for r in val_rows}, indent=2, default=float),\\n        encoding=\\"utf-8\\",\\n    )\\n\\n    best_q = min(val_rows, key=rank_key)\\n    best_classical_paired = max(classical_val_rows, key=lambda r: r[\\"average_precision\\"])\\n    selection = {\\n        \\"best_qsvm_id\\": best_q[\\"id\\"],\\n        \\"best_qsvm_val_auprc\\": best_q[\\"average_precision\\"],\\n        \\"best_classical_paired\\": best_classical_paired[\\"model\\"],\\n        \\"best_classical_paired_pca\\": best_classical_paired[\\"pca_components\\"],\\n        \\"best_classical_paired_val_auprc\\": best_classical_paired[\\"average_precision\\"],\\n        \\"control_id\\": \\"Q10\\",\\n        \\"seed\\": seed,\\n        \\"note\\": \\"Selection used validation only; test not consulted.\\",\\n    }\\n    (paths[\\"ablation\\"] / \\"selection.json\\").write_text(json.dumps(selection, indent=2), encoding=\\"utf-8\\")\\n\\n    result: dict[str, Any] = {\\n        \\"selection\\": selection,\\n        \\"validation\\": val_df,\\n        \\"paths\\": {k: str(v) for k, v in paths.items()},\\n    }\\n\\n    if skip_test:\\n        return result\\n\\n    # ---- Final test: original control, best QSVM, best paired classical, dummy ----\\n    test_rows: list[dict[str, Any]] = []\\n    final_ids = list(dict.fromkeys([best_q[\\"id\\"], \\"Q10\\"]))  # unique order\\n\\n    for cfg_id in final_ids:\\n        qcfg = next(c for c in abl_cfg[\\"quantum_configs\\"] if c[\\"id\\"] == cfg_id)\\n        n = int(qcfg[\\"pca_components\\"])\\n        part, _ = apply_angular(pca_cache[n], qcfg[\\"angular_scaler\\"], seed=seed)\\n        kpaths = _kernel_paths(paths[\\"kernels\\"], cfg_id, seed)\\n        K_train = np.load(kpaths[\\"train\\"])\\n        K_test = np.load(kpaths[\\"test\\"])\\n        m = evaluate_precomputed_svm(\\n            K_train, K_test, part.y_train, part.y_test, seed=seed, model_name=f\\"qsvm_{cfg_id}\\"\\n        )\\n        diag = diagnose_kernel(K_train, part.y_train)\\n        test_rows.append(\\n            {\\n                **m,\\n                \\"id\\": cfg_id,\\n                \\"pca_components\\": n,\\n                \\"angular_scaler\\": qcfg[\\"angular_scaler\\"],\\n                \\"feature_map\\": qcfg[\\"feature_map\\"],\\n                \\"reps\\": qcfg[\\"reps\\"],\\n                \\"alignment\\": diag.get(\\"alignment\\"),\\n                \\"effective_rank\\": diag.get(\\"effective_rank\\"),\\n                \\"delta_k\\": diag.get(\\"delta_k\\"),\\n                \\"split\\": \\"test\\",\\n                \\"family\\": \\"qsvm\\",\\n            }\\n        )\\n\\n    # Best paired classical on test\\n    n_best = int(best_classical_paired[\\"pca_components\\"])\\n    part_c = pca_cache[n_best]\\n    for row in evaluate_classical_on_embedding(part_c, seed=seed, split=\\"test\\"):\\n        if row[\\"model\\"] in {best_classical_paired[\\"model\\"], \\"dummy\\"}:\\n            test_rows.append(\\n                {\\n                    **row,\\n                    \\"id\\": f\\"classical_{row[\'model\']}_pca{n_best}\\",\\n                    \\"pca_components\\": n_best,\\n                    \\"angular_scaler\\": \\"none\\",\\n                    \\"split\\": \\"test\\",\\n                    \\"family\\": \\"classical_paired\\",\\n                }\\n            )\\n\\n    # Bootstrap delta: best QSVM vs best classical paired on test\\n    best_q_test = next(r for r in test_rows if r.get(\\"id\\") == best_q[\\"id\\"])\\n    best_c_test = next(\\n        r\\n        for r in test_rows\\n        if r.get(\\"family\\") == \\"classical_paired\\" and r[\\"model\\"] == best_classical_paired[\\"model\\"]\\n    )\\n    # Need scores for bootstrap — recompute quickly\\n    qcfg_b = next(c for c in abl_cfg[\\"quantum_configs\\"] if c[\\"id\\"] == best_q[\\"id\\"])\\n    n_b = int(qcfg_b[\\"pca_components\\"])\\n    part_b, _ = apply_angular(pca_cache[n_b], qcfg_b[\\"angular_scaler\\"], seed=seed)\\n    K_train_b = np.load(_kernel_paths(paths[\\"kernels\\"], best_q[\\"id\\"], seed)[\\"train\\"])\\n    K_test_b = np.load(_kernel_paths(paths[\\"kernels\\"], best_q[\\"id\\"], seed)[\\"test\\"])\\n    clf_q = SVC(kernel=\\"precomputed\\", class_weight=\\"balanced\\", random_state=seed)\\n    clf_q.fit(K_train_b, part_b.y_train)\\n    q_scores = 1.0 / (1.0 + np.exp(-clf_q.decision_function(K_test_b)))\\n\\n    model_c = {\\n        \\"dummy\\": make_dummy(seed=seed),\\n        \\"logistic\\": make_logistic(seed=seed),\\n        \\"svm_linear\\": make_linear_svm(seed=seed),\\n        \\"svm_rbf\\": make_rbf_svm(seed=seed),\\n        \\"svm_poly\\": make_poly_svm(seed=seed),\\n        \\"knn\\": make_knn(),\\n    }[best_classical_paired[\\"model\\"]]\\n    part_c = pca_cache[n_best]\\n    model_c.fit(part_c.X_train, part_c.y_train)\\n    c_scores = predict_proba_positive(model_c, part_c.X_test)\\n    assert c_scores is not None\\n\\n    boot = bootstrap_auprc_delta(\\n        part_c.y_test,\\n        q_scores,\\n        c_scores,\\n        n_boot=int(abl_cfg[\\"evaluation\\"][\\"bootstrap_iterations\\"]),\\n        seed=seed,\\n    )\\n    classification = classify_result(\\n        delta_mean=boot[\\"delta_mean\\"],\\n        ci_low=boot[\\"ci_low\\"],\\n        ci_high=boot[\\"ci_high\\"],\\n        min_delta=float(abl_cfg[\\"evaluation\\"][\\"min_auprc_delta\\"]),\\n        q_auprc=best_q_test[\\"average_precision\\"],\\n        prevalence=float(np.mean(part_c.y_test)),\\n        alignment=float(best_q.get(\\"alignment\\") or 0.0),\\n        delta_k=float(best_q.get(\\"delta_k\\") or 0.0),\\n    )\\n\\n    test_df = pd.DataFrame(test_rows)\\n    test_df.to_csv(paths[\\"ablation\\"] / \\"test_final.csv\\", index=False)\\n    summary = {\\n        \\"selection\\": selection,\\n        \\"bootstrap\\": boot,\\n        \\"classification\\": classification,\\n        \\"best_qsvm_test\\": best_q_test,\\n        \\"best_classical_test\\": best_c_test,\\n    }\\n    (paths[\\"ablation\\"] / \\"final_summary.json\\").write_text(\\n        json.dumps(summary, indent=2, default=float), encoding=\\"utf-8\\"\\n    )\\n    result[\\"test\\"] = test_df\\n    result[\\"summary\\"] = summary\\n    return result\\n\\n\\ndef bootstrap_auprc_delta(\\n    y_true: np.ndarray,\\n    scores_a: np.ndarray,\\n    scores_b: np.ndarray,\\n    n_boot: int = 1000,\\n    seed: int = 42,\\n    confidence: float = 0.95,\\n) -> dict[str, float]:\\n    from sklearn.metrics import average_precision_score\\n\\n    rng = np.random.default_rng(seed)\\n    y_true = np.asarray(y_true)\\n    n = len(y_true)\\n    deltas = []\\n    for _ in range(n_boot):\\n        idx = rng.integers(0, n, size=n)\\n        if len(np.unique(y_true[idx])) < 2:\\n            continue\\n        a = average_precision_score(y_true[idx], scores_a[idx])\\n        b = average_precision_score(y_true[idx], scores_b[idx])\\n        deltas.append(a - b)\\n    deltas_arr = np.asarray(deltas)\\n    alpha = (1 - confidence) / 2\\n    return {\\n        \\"delta_mean\\": float(np.mean(deltas_arr)),\\n        \\"delta_std\\": float(np.std(deltas_arr)),\\n        \\"ci_low\\": float(np.quantile(deltas_arr, alpha)),\\n        \\"ci_high\\": float(np.quantile(deltas_arr, 1 - alpha)),\\n        \\"n_boot_effective\\": len(deltas_arr),\\n    }\\n\\n\\ndef classify_result(\\n    delta_mean: float,\\n    ci_low: float,\\n    ci_high: float,\\n    min_delta: float,\\n    q_auprc: float,\\n    prevalence: float,\\n    alignment: float,\\n    delta_k: float,\\n) -> dict[str, str]:\\n    \\"\\"\\"Automatic interpretation labels from the revised plan (no \'quantum advantage\' wording).\\"\\"\\"\\n    near_dummy = abs(q_auprc - prevalence) < 0.03 and abs(alignment) < 0.05 and abs(delta_k) < 0.02\\n    if near_dummy:\\n        label = \\"QUANTUM_KERNEL_NOT_INFORMATIVE\\"\\n        conclusion = (\\n            \\"O baixo desempenho não decorreu apenas da ausência de normalização angular. \\"\\n            \\"O feature map testado não representa adequadamente a estrutura preditiva da tarefa.\\"\\n        )\\n    elif ci_low > 0 and delta_mean >= min_delta:\\n        label = \\"CANDIDATE_PREDICTIVE_QUANTUM_GAIN\\"\\n        conclusion = (\\n            \\"QSVM superou o baseline clássico pareado no teste com intervalo bootstrap da \\"\\n            \\"diferença acima de zero. Resultado preliminar — não interpretar como vantagem \\"\\n            \\"computacional quântica geral.\\"\\n        )\\n    elif delta_mean > 0 and ci_low <= 0:\\n        label = \\"NON_GENERALIZING_QUANTUM_GAIN\\"\\n        conclusion = (\\n            \\"Há sinal positivo pontual, mas o intervalo bootstrap da diferença inclui zero \\"\\n            \\"(ganho não generaliza de forma estável).\\"\\n        )\\n    elif q_auprc > prevalence + 0.03 and delta_mean <= 0:\\n        label = \\"BETTER_QUANTUM_REPRESENTATION_WITHOUT_ADVANTAGE\\"\\n        conclusion = (\\n            \\"A adaptação angular tornou o kernel mais informativo que o controle/Dummy, \\"\\n            \\"mas o ganho não foi suficiente para superar kernels clássicos equivalentes.\\"\\n        )\\n    else:\\n        label = \\"QUANTUM_KERNEL_NOT_INFORMATIVE\\"\\n        conclusion = (\\n            \\"Nenhuma configuração selecionada produziu evidência preditiva estável \\"\\n            \\"frente aos baselines clássicos pareados.\\"\\n        )\\n    return {\\"label\\": label, \\"conclusion\\": conclusion}\\n\\n\\nif __name__ == \\"__main__\\":\\n    logging.basicConfig(level=logging.INFO, format=\\"%(levelname)s %(message)s\\")\\n    out = run_ablation()\\n    print(json.dumps(out.get(\\"summary\\", out.get(\\"selection\\")), indent=2, default=float))\\n", "reporting/__init__.py": "\\"\\"\\"Reporting helpers.\\"\\"\\"\\n\\nfrom qml_air_quality.reporting.ablation_report import (\\n    plot_ablation_figures,\\n    write_ablation_report,\\n)\\nfrom qml_air_quality.reporting.fair_benchmark_plots import generate_fair_benchmark_plots\\n\\n__all__ = [\\n    \\"generate_fair_benchmark_plots\\",\\n    \\"plot_ablation_figures\\",\\n    \\"write_ablation_report\\",\\n]\\n", "reporting/fair_benchmark_plots.py": "\\"\\"\\"Plots for the Farooq-style fair benchmark artifacts.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport matplotlib\\n\\nmatplotlib.use(\\"Agg\\")\\nimport matplotlib.pyplot as plt\\nimport numpy as np\\nimport pandas as pd\\nfrom sklearn.metrics import (\\n    PrecisionRecallDisplay,\\n    RocCurveDisplay,\\n    confusion_matrix,\\n)\\n\\nFAMILY_COLORS = {\\n    \\"classical_full\\": \\"#4C78A8\\",\\n    \\"classical_paired\\": \\"#F58518\\",\\n    \\"qsvm\\": \\"#54A24B\\",\\n}\\n\\n\\ndef _plots_dir(artifacts_dir: Path) -> Path:\\n    out = artifacts_dir / \\"plots\\"\\n    out.mkdir(parents=True, exist_ok=True)\\n    return out\\n\\n\\ndef _load_summary(artifacts_dir: Path) -> pd.DataFrame | None:\\n    path = artifacts_dir / \\"final_metrics_summary.csv\\"\\n    if not path.exists():\\n        return None\\n    return pd.read_csv(path)\\n\\n\\ndef _prediction_csvs(artifacts_dir: Path) -> list[Path]:\\n    pred_dir = artifacts_dir / \\"predictions\\"\\n    if not pred_dir.exists():\\n        return []\\n    return sorted(pred_dir.glob(\\"*.csv\\"))\\n\\n\\ndef _parse_pred_stem(stem: str) -> dict[str, Any] | None:\\n    \\"\\"\\"Parse \'{model}_{seed}_{train_size}\' with model possibly containing underscores.\\"\\"\\"\\n    parts = stem.rsplit(\\"_\\", 2)\\n    if len(parts) != 3:\\n        return None\\n    model, seed_s, n_s = parts\\n    try:\\n        return {\\"model\\": model, \\"seed\\": int(seed_s), \\"train_size\\": int(n_s)}\\n    except ValueError:\\n        return None\\n\\n\\ndef _mean_scores_by_model(\\n    artifacts_dir: Path, train_size: int | None = None\\n) -> dict[str, pd.DataFrame]:\\n    \\"\\"\\"Average scores across seeds on the fixed test set (same row order).\\"\\"\\"\\n    buckets: dict[str, list[pd.DataFrame]] = {}\\n    for path in _prediction_csvs(artifacts_dir):\\n        meta = _parse_pred_stem(path.stem)\\n        if meta is None:\\n            continue\\n        if train_size is not None and meta[\\"train_size\\"] != train_size:\\n            continue\\n        df = pd.read_csv(path)\\n        key = f\\"{meta[\'model\']}|{meta[\'train_size\']}\\"\\n        buckets.setdefault(key, []).append(df)\\n\\n    out: dict[str, pd.DataFrame] = {}\\n    for key, frames in buckets.items():\\n        base = frames[0][[\\"timestamp\\", \\"y_true\\", \\"original_index\\"]].copy() if \\"original_index\\" in frames[0] else frames[0][[\\"timestamp\\", \\"y_true\\"]].copy()\\n        scores = np.mean([f[\\"score\\"].to_numpy(dtype=float) for f in frames], axis=0)\\n        # majority / mean-threshold pred from mean score vs first threshold\\n        thr = float(frames[0][\\"threshold\\"].iloc[0])\\n        base[\\"score\\"] = scores\\n        base[\\"y_pred\\"] = (scores >= thr).astype(int)\\n        base[\\"threshold\\"] = thr\\n        base[\\"model\\"] = key.split(\\"|\\")[0]\\n        base[\\"train_size\\"] = int(key.split(\\"|\\")[1])\\n        out[key] = base\\n    return out\\n\\n\\ndef plot_auprc_bars(artifacts_dir: Path) -> Path | None:\\n    summary = _load_summary(artifacts_dir)\\n    if summary is None or summary.empty:\\n        return None\\n    plots = _plots_dir(artifacts_dir)\\n    train_sizes = sorted(summary[\\"train_size\\"].unique())\\n    n = len(train_sizes)\\n    fig, axes = plt.subplots(1, n, figsize=(5.2 * n, 4.2), sharey=False)\\n    if n == 1:\\n        axes = [axes]\\n\\n    for ax, ts in zip(axes, train_sizes):\\n        sub = summary[summary[\\"train_size\\"] == ts].sort_values(\\"average_precision_mean\\")\\n        colors = [FAMILY_COLORS.get(f, \\"#999999\\") for f in sub[\\"family\\"]]\\n        y = np.arange(len(sub))\\n        err = sub[\\"average_precision_std\\"].fillna(0).to_numpy()\\n        ax.barh(y, sub[\\"average_precision_mean\\"], xerr=err, color=colors, capsize=3)\\n        ax.set_yticks(y)\\n        ax.set_yticklabels(sub[\\"model\\"])\\n        ax.set_xlabel(\\"AUPRC (teste)\\")\\n        ax.set_title(f\\"train_size={ts}\\")\\n        ax.set_xlim(0, max(0.05, float(sub[\\"average_precision_mean\\"].max()) * 1.15))\\n\\n    fig.suptitle(\\"Fair benchmark — AUPRC médio (± std entre sementes)\\", y=1.02)\\n    fig.tight_layout()\\n    out = plots / \\"auprc_bars.png\\"\\n    fig.savefig(out, dpi=150, bbox_inches=\\"tight\\")\\n    plt.close(fig)\\n    return out\\n\\n\\ndef plot_pr_roc(artifacts_dir: Path) -> Path | None:\\n    summary = _load_summary(artifacts_dir)\\n    if summary is None:\\n        return None\\n    train_size = int(summary[\\"train_size\\"].max())\\n    scored = _mean_scores_by_model(artifacts_dir, train_size=train_size)\\n    if not scored:\\n        return None\\n\\n    plots = _plots_dir(artifacts_dir)\\n    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))\\n    for key, df in scored.items():\\n        name = key.split(\\"|\\")[0]\\n        y = df[\\"y_true\\"].to_numpy()\\n        s = df[\\"score\\"].to_numpy()\\n        if len(np.unique(y)) < 2:\\n            continue\\n        PrecisionRecallDisplay.from_predictions(y, s, name=name, ax=axes[0])\\n        RocCurveDisplay.from_predictions(y, s, name=name, ax=axes[1])\\n    axes[0].set_title(f\\"Precision–Recall (n={train_size}, scores médios)\\")\\n    axes[1].set_title(f\\"ROC (n={train_size}, scores médios)\\")\\n    axes[0].legend(loc=\\"best\\", fontsize=8)\\n    axes[1].legend(loc=\\"best\\", fontsize=8)\\n    fig.tight_layout()\\n    out = plots / \\"pr_roc_curves.png\\"\\n    fig.savefig(out, dpi=150, bbox_inches=\\"tight\\")\\n    plt.close(fig)\\n    return out\\n\\n\\ndef plot_confusion_matrices(artifacts_dir: Path, max_models: int = 6) -> Path | None:\\n    summary = _load_summary(artifacts_dir)\\n    if summary is None:\\n        return None\\n    train_size = int(summary[\\"train_size\\"].max())\\n    scored = _mean_scores_by_model(artifacts_dir, train_size=train_size)\\n    if not scored:\\n        return None\\n\\n    # Prefer qsvm + paired + best full by AUPRC\\n    ranking = summary[summary[\\"train_size\\"] == train_size].sort_values(\\n        \\"average_precision_mean\\", ascending=False\\n    )\\n    wanted = []\\n    for _, row in ranking.iterrows():\\n        key = f\\"{row[\'model\']}|{train_size}\\"\\n        if key in scored and key not in wanted:\\n            wanted.append(key)\\n        if len(wanted) >= max_models:\\n            break\\n\\n    n = len(wanted)\\n    if n == 0:\\n        return None\\n    cols = min(3, n)\\n    rows = int(np.ceil(n / cols))\\n    fig, axes = plt.subplots(rows, cols, figsize=(4.0 * cols, 3.6 * rows))\\n    axes_list = np.atleast_1d(axes).ravel()\\n    for ax, key in zip(axes_list, wanted):\\n        df = scored[key]\\n        cm = confusion_matrix(df[\\"y_true\\"], df[\\"y_pred\\"], labels=[0, 1])\\n        im = ax.imshow(cm, cmap=\\"Blues\\")\\n        ax.set_xticks([0, 1])\\n        ax.set_yticks([0, 1])\\n        ax.set_xticklabels([\\"pred 0\\", \\"pred 1\\"])\\n        ax.set_yticklabels([\\"true 0\\", \\"true 1\\"])\\n        for (i, j), v in np.ndenumerate(cm):\\n            ax.text(j, i, str(v), ha=\\"center\\", va=\\"center\\", color=\\"black\\")\\n        ax.set_title(key.split(\\"|\\")[0], fontsize=9)\\n        fig.colorbar(im, ax=ax, fraction=0.046)\\n    for ax in axes_list[n:]:\\n        ax.axis(\\"off\\")\\n    fig.suptitle(f\\"Matrizes de confusão (limiar F2 congelado, n={train_size})\\", y=1.02)\\n    fig.tight_layout()\\n    out = _plots_dir(artifacts_dir) / \\"confusion_matrices.png\\"\\n    fig.savefig(out, dpi=150, bbox_inches=\\"tight\\")\\n    plt.close(fig)\\n    return out\\n\\n\\ndef plot_bootstrap_deltas(artifacts_dir: Path) -> Path | None:\\n    path = artifacts_dir / \\"pairwise_bootstrap.csv\\"\\n    if not path.exists() or path.stat().st_size < 2:\\n        return None\\n    boot = pd.read_csv(path)\\n    if boot.empty or \\"delta_auprc_mean\\" not in boot.columns:\\n        return None\\n\\n    plots = _plots_dir(artifacts_dir)\\n    fig, ax = plt.subplots(figsize=(8, max(2.5, 0.55 * len(boot) + 1)))\\n    y = np.arange(len(boot))\\n    means = boot[\\"delta_auprc_mean\\"].to_numpy()\\n    lo = boot[\\"delta_auprc_ci_low\\"].to_numpy()\\n    hi = boot[\\"delta_auprc_ci_high\\"].to_numpy()\\n    for yi, mean, lo_i, hi_i in zip(y, means, lo, hi):\\n        if mean > 0 and lo_i > 0:\\n            color = \\"#54A24B\\"\\n        elif mean < 0 and hi_i < 0:\\n            color = \\"#E45756\\"\\n        else:\\n            color = \\"#999999\\"\\n        ax.errorbar(\\n            mean,\\n            yi,\\n            xerr=[[mean - lo_i], [hi_i - mean]],\\n            fmt=\\"o\\",\\n            color=color,\\n            ecolor=color,\\n            capsize=4,\\n        )\\n    ax.axvline(0, color=\\"gray\\", ls=\\"--\\", lw=1)\\n    labels = [f\\"{r.get(\'comparison\', \'\')}\\"[:70] for _, r in boot.iterrows()]\\n    ax.set_yticks(y)\\n    ax.set_yticklabels(labels, fontsize=8)\\n    ax.set_xlabel(\\"ΔAUPRC (QSVM − baseline) com IC 95%\\")\\n    ax.set_title(\\"Block bootstrap — diferenças pareadas\\")\\n    fig.tight_layout()\\n    out = plots / \\"bootstrap_deltas.png\\"\\n    fig.savefig(out, dpi=150, bbox_inches=\\"tight\\")\\n    plt.close(fig)\\n    return out\\n\\n\\ndef plot_learning_curves(artifacts_dir: Path) -> Path | None:\\n    summary = _load_summary(artifacts_dir)\\n    if summary is None:\\n        return None\\n    sizes = sorted(summary[\\"train_size\\"].unique())\\n    if len(sizes) < 2:\\n        return None\\n    plots = _plots_dir(artifacts_dir)\\n    fig, ax = plt.subplots(figsize=(7.5, 4.5))\\n    for model, g in summary.groupby(\\"model\\"):\\n        g = g.sort_values(\\"train_size\\")\\n        fam = g[\\"family\\"].iloc[0]\\n        ax.errorbar(\\n            g[\\"train_size\\"],\\n            g[\\"average_precision_mean\\"],\\n            yerr=g[\\"average_precision_std\\"].fillna(0),\\n            marker=\\"o\\",\\n            label=model,\\n            color=FAMILY_COLORS.get(fam, None),\\n            capsize=3,\\n        )\\n    ax.set_xlabel(\\"Tamanho do treino\\")\\n    ax.set_ylabel(\\"AUPRC (teste)\\")\\n    ax.set_title(\\"Curva amostral — eficiência com n=200 vs 500\\")\\n    ax.legend(fontsize=7, loc=\\"best\\")\\n    fig.tight_layout()\\n    out = plots / \\"learning_curves.png\\"\\n    fig.savefig(out, dpi=150, bbox_inches=\\"tight\\")\\n    plt.close(fig)\\n    return out\\n\\n\\ndef plot_selection_quantum(artifacts_dir: Path) -> Path | None:\\n    path = artifacts_dir / \\"model_selection.csv\\"\\n    if not path.exists():\\n        return None\\n    sel = pd.read_csv(path)\\n    q = sel[sel[\\"model_id\\"].astype(str).str.startswith(\\"qsvm\\")].copy()\\n    if q.empty:\\n        return None\\n    agg = (\\n        q.groupby(\\"model_id\\", as_index=False)\\n        .agg(val_auprc=(\\"val_auprc\\", \\"mean\\"), val_f2=(\\"val_f2\\", \\"mean\\"))\\n        .sort_values(\\"val_auprc\\")\\n    )\\n    plots = _plots_dir(artifacts_dir)\\n    fig, ax = plt.subplots(figsize=(7, 3.8))\\n    y = np.arange(len(agg))\\n    ax.barh(y, agg[\\"val_auprc\\"], color=\\"#54A24B\\")\\n    ax.set_yticks(y)\\n    ax.set_yticklabels(agg[\\"model_id\\"])\\n    ax.set_xlabel(\\"AUPRC (validação, média das sementes)\\")\\n    ax.set_title(\\"Seleção quântica Q01–Q04 (só validação)\\")\\n    fig.tight_layout()\\n    out = plots / \\"quantum_selection.png\\"\\n    fig.savefig(out, dpi=150, bbox_inches=\\"tight\\")\\n    plt.close(fig)\\n    return out\\n\\n\\ndef plot_kernel_heatmap(artifacts_dir: Path) -> Path | None:\\n    kernels = sorted((artifacts_dir / \\"kernels\\").glob(\\"final_*_K_train.npy\\"))\\n    if not kernels:\\n        kernels = sorted((artifacts_dir / \\"kernels\\").glob(\\"Q*_K_train.npy\\"))\\n    if not kernels:\\n        return None\\n    K = np.load(kernels[0])\\n    plots = _plots_dir(artifacts_dir)\\n    fig, ax = plt.subplots(figsize=(5, 4.2))\\n    im = ax.imshow(K, cmap=\\"viridis\\", aspect=\\"auto\\")\\n    ax.set_title(kernels[0].stem)\\n    ax.set_xlabel(\\"amostra\\")\\n    ax.set_ylabel(\\"amostra\\")\\n    fig.colorbar(im, ax=ax, fraction=0.046)\\n    fig.tight_layout()\\n    out = plots / \\"kernel_heatmap.png\\"\\n    fig.savefig(out, dpi=150, bbox_inches=\\"tight\\")\\n    plt.close(fig)\\n    return out\\n\\n\\ndef generate_fair_benchmark_plots(artifacts_dir: str | Path) -> list[Path]:\\n    \\"\\"\\"Build the standard plot panel from saved fair-benchmark artifacts.\\"\\"\\"\\n    artifacts_dir = Path(artifacts_dir)\\n    generators = [\\n        plot_auprc_bars,\\n        plot_pr_roc,\\n        plot_confusion_matrices,\\n        plot_bootstrap_deltas,\\n        plot_learning_curves,\\n        plot_selection_quantum,\\n        plot_kernel_heatmap,\\n    ]\\n    written: list[Path] = []\\n    for fn in generators:\\n        path = fn(artifacts_dir)\\n        if path is not None:\\n            written.append(path)\\n    return written\\n", "reporting/ablation_report.py": "\\"\\"\\"Ablation report and plots.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\n\\nimport matplotlib.pyplot as plt\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom qml_air_quality.config import project_root\\n\\n\\ndef write_ablation_report(\\n    ablation_dir: str | Path | None = None,\\n    out_path: str | Path | None = None,\\n) -> Path:\\n    root = project_root()\\n    abl = Path(ablation_dir) if ablation_dir else root / \\"artifacts\\" / \\"ablation\\"\\n    out = Path(out_path) if out_path else abl / \\"reports\\" / \\"ablation_report.md\\"\\n    out.parent.mkdir(parents=True, exist_ok=True)\\n\\n    val = pd.read_csv(abl / \\"validation_ranking.csv\\") if (abl / \\"validation_ranking.csv\\").exists() else None\\n    test = pd.read_csv(abl / \\"test_final.csv\\") if (abl / \\"test_final.csv\\").exists() else None\\n    selection = json.loads((abl / \\"selection.json\\").read_text()) if (abl / \\"selection.json\\").exists() else {}\\n    summary = json.loads((abl / \\"final_summary.json\\").read_text()) if (abl / \\"final_summary.json\\").exists() else {}\\n\\n    lines = [\\n        \\"# Relatório de ablação — escala angular e kernels quânticos\\",\\n        \\"\\",\\n        \\"## Contexto\\",\\n        \\"\\",\\n        \\"Tarefa temporal sem vazamento: prever se PM2.5 em t+24h excede o P90 do **treino**.\\",\\n        \\"Esta iteração testa se escala angular, dimensionalidade e profundidade do feature map\\",\\n        \\"tornam o kernel quântico mais informativo — **sem** reproduzir a classificação circular\\",\\n        \\"de AQI de Farooq et al. (2024).\\",\\n        \\"\\",\\n        \\"## Seleção (somente validação)\\",\\n        \\"\\",\\n        \\"```json\\",\\n        json.dumps(selection, indent=2),\\n        \\"```\\",\\n        \\"\\",\\n    ]\\n    if val is not None:\\n        cols = [\\n            c\\n            for c in [\\n                \\"id\\",\\n                \\"pca_components\\",\\n                \\"angular_scaler\\",\\n                \\"feature_map\\",\\n                \\"reps\\",\\n                \\"average_precision\\",\\n                \\"recall_extreme\\",\\n                \\"alignment\\",\\n                \\"effective_rank\\",\\n                \\"delta_k\\",\\n                \\"kernel_seconds\\",\\n            ]\\n            if c in val.columns\\n        ]\\n        lines += [\\n            \\"## Ranking de validação\\",\\n            \\"\\",\\n            \\"```\\",\\n            val[cols].to_string(index=False),\\n            \\"```\\",\\n            \\"\\",\\n        ]\\n    if test is not None:\\n        lines += [\\n            \\"## Teste final (uma vez)\\",\\n            \\"\\",\\n            \\"```\\",\\n            test.to_string(index=False),\\n            \\"```\\",\\n            \\"\\",\\n        ]\\n    if summary:\\n        lines += [\\n            \\"## Classificação automática\\",\\n            \\"\\",\\n            f\\"- Rótulo: `{summary.get(\'classification\', {}).get(\'label\')}`\\",\\n            f\\"- Conclusão: {summary.get(\'classification\', {}).get(\'conclusion\')}\\",\\n            \\"\\",\\n            \\"### Bootstrap da diferença (QSVM − clássico pareado)\\",\\n            \\"\\",\\n            \\"```json\\",\\n            json.dumps(summary.get(\\"bootstrap\\", {}), indent=2),\\n            \\"```\\",\\n            \\"\\",\\n        ]\\n    lines += [\\n        \\"## Comparação metodológica com Farooq et al.\\",\\n        \\"\\",\\n        \\"| Aspecto | Farooq et al. | Esta PoC |\\",\\n        \\"|---|---|---|\\",\\n        \\"| Alvo | AQI contemporâneo (faixas) | Extremo PM2.5 em t+24h |\\",\\n        \\"| Features | PM2.5 + temperatura (~2D) | lags/rolling/meteo → PCA |\\",\\n        \\"| Escala | MinMax [0,1] | ablação none/[0,1]/[0,π]/[-π,π]/quantile |\\",\\n        \\"| Métrica | accuracy | **AUPRC** |\\",\\n        \\"| Split | 80/20 (não temporal explícito) | temporal 60/20/20 |\\",\\n        \\"\\",\\n        \\"## Limitações\\",\\n        \\"\\",\\n        \\"- Uma seed principal na execução padrão.\\",\\n        \\"- Simulador ideal (sem ruído de hardware).\\",\\n        \\"- Subamostra finita para viabilidade do kernel.\\",\\n        \\"- Melhoria estrutural do kernel ≠ vantagem computacional quântica.\\",\\n        \\"\\",\\n    ]\\n    out.write_text(\\"\\\\n\\".join(lines), encoding=\\"utf-8\\")\\n    return out\\n\\n\\ndef plot_ablation_figures(ablation_dir: str | Path | None = None) -> list[Path]:\\n    root = project_root()\\n    abl = Path(ablation_dir) if ablation_dir else root / \\"artifacts\\" / \\"ablation\\"\\n    plots = abl / \\"plots\\"\\n    plots.mkdir(parents=True, exist_ok=True)\\n    written: list[Path] = []\\n\\n    val_path = abl / \\"validation_ranking.csv\\"\\n    if not val_path.exists():\\n        return written\\n    val = pd.read_csv(val_path)\\n\\n    fig, ax = plt.subplots(figsize=(9, 4))\\n    order = val.sort_values(\\"average_precision\\")\\n    ax.barh(order[\\"id\\"], order[\\"average_precision\\"])\\n    ax.set_xlabel(\\"AUPRC (validação)\\")\\n    ax.set_title(\\"AUPRC de validação por configuração\\")\\n    fig.tight_layout()\\n    p = plots / \\"validation_auprc_by_configuration.png\\"\\n    fig.savefig(p, dpi=120)\\n    plt.close(fig)\\n    written.append(p)\\n\\n    if \\"alignment\\" in val.columns:\\n        fig, ax = plt.subplots(figsize=(9, 4))\\n        order = val.sort_values(\\"alignment\\")\\n        ax.barh(order[\\"id\\"], order[\\"alignment\\"])\\n        ax.set_xlabel(\\"Kernel-target alignment (treino)\\")\\n        ax.set_title(\\"Alinhamento kernel–rótulo\\")\\n        fig.tight_layout()\\n        p = plots / \\"kernel_alignment_comparison.png\\"\\n        fig.savefig(p, dpi=120)\\n        plt.close(fig)\\n        written.append(p)\\n\\n    if \\"effective_rank\\" in val.columns:\\n        fig, ax = plt.subplots(figsize=(9, 4))\\n        order = val.sort_values(\\"effective_rank\\")\\n        ax.barh(order[\\"id\\"], order[\\"effective_rank\\"])\\n        ax.set_xlabel(\\"Effective rank\\")\\n        ax.set_title(\\"Posto efetivo dos kernels\\")\\n        fig.tight_layout()\\n        p = plots / \\"kernel_effective_rank.png\\"\\n        fig.savefig(p, dpi=120)\\n        plt.close(fig)\\n        written.append(p)\\n\\n    if \\"delta_k\\" in val.columns:\\n        fig, ax = plt.subplots(figsize=(9, 4))\\n        order = val.sort_values(\\"delta_k\\")\\n        ax.barh(order[\\"id\\"], order[\\"delta_k\\"])\\n        ax.axvline(0, color=\\"gray\\", lw=0.8)\\n        ax.set_xlabel(\\"ΔK (same − different)\\")\\n        ax.set_title(\\"Contraste entre classes no kernel\\")\\n        fig.tight_layout()\\n        p = plots / \\"kernel_contrast_same_vs_different.png\\"\\n        fig.savefig(p, dpi=120)\\n        plt.close(fig)\\n        written.append(p)\\n\\n    # Kernel heatmaps for all cached configs\\n    kernels = abl / \\"kernels\\"\\n    mats = sorted(kernels.glob(\\"*_seed*_K_train.npy\\"))\\n    if mats:\\n        n = len(mats)\\n        cols = min(5, n)\\n        rows = int(np.ceil(n / cols))\\n        fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))\\n        axes_arr = np.atleast_1d(axes).ravel()\\n        for ax, path in zip(axes_arr, mats):\\n            K = np.load(path)\\n            ax.imshow(K, cmap=\\"viridis\\", aspect=\\"auto\\")\\n            ax.set_title(path.name.split(\\"_seed\\")[0], fontsize=8)\\n            ax.set_xticks([])\\n            ax.set_yticks([])\\n        for ax in axes_arr[len(mats) :]:\\n            ax.axis(\\"off\\")\\n        fig.suptitle(\\"Matrizes de kernel (treino)\\")\\n        fig.tight_layout()\\n        p = plots / \\"kernel_matrices_by_configuration.png\\"\\n        fig.savefig(p, dpi=120)\\n        plt.close(fig)\\n        written.append(p)\\n\\n        fig, ax = plt.subplots(figsize=(8, 4))\\n        for path in mats:\\n            K = np.load(path)\\n            off = K[~np.eye(K.shape[0], dtype=bool)]\\n            ax.hist(off, bins=40, alpha=0.35, label=path.name.split(\\"_seed\\")[0], density=True)\\n        ax.set_xlabel(\\"K_ij (i≠j)\\")\\n        ax.set_title(\\"Histogramas dos valores fora da diagonal\\")\\n        ax.legend(fontsize=7, ncol=2)\\n        fig.tight_layout()\\n        p = plots / \\"kernel_value_histograms.png\\"\\n        fig.savefig(p, dpi=120)\\n        plt.close(fig)\\n        written.append(p)\\n\\n    test_path = abl / \\"test_final.csv\\"\\n    if test_path.exists():\\n        test = pd.read_csv(test_path)\\n        fig, ax = plt.subplots(figsize=(8, 4))\\n        ax.barh(test[\\"id\\"].astype(str), test[\\"average_precision\\"])\\n        ax.set_xlabel(\\"AUPRC (teste)\\")\\n        ax.set_title(\\"QSVM vs clássicos pareados (teste)\\")\\n        fig.tight_layout()\\n        p = plots / \\"qsvm_vs_classical_paired.png\\"\\n        fig.savefig(p, dpi=120)\\n        plt.close(fig)\\n        written.append(p)\\n\\n    return written\\n"}')

written = 0
for rel, content in PACKAGE_FILES.items():
    path = PKG_ROOT / "qml_air_quality" / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
    written += 1

sys.path.insert(0, str(PKG_ROOT))
print(f"ON_KAGGLE={ON_KAGGLE}")
print(f"WORK={WORK}")
print(f"pacote: {written} arquivos em {PKG_ROOT / 'qml_air_quality'}")


## 3. Configuração (small / medium / large)


In [ ]:
from __future__ import annotations

import logging
from pathlib import Path

import pandas as pd

# ==========================================================
# MODO: 'small' | 'medium' | 'large'
# ==========================================================
MODE = "small"

STAGE = "all"  # "selection" | "final" | "all"
SKIP_QUANTUM = False
VERBOSE = True

PRESETS = {
    "small": {
        "train_sizes": [80],
        "validation_size": 40,
        "test_size": 40,
        "selection_seeds": [42],
        "final_seeds": [42],
        "bootstrap_iterations": 200,
        "artifacts_subdir": "farooq_fair_benchmark_small",
    },
    "medium": {
        "train_sizes": [200],
        "validation_size": 100,
        "test_size": 150,
        "selection_seeds": [7, 11, 13, 21, 42],
        "final_seeds": [7, 11, 13, 21, 42],
        "bootstrap_iterations": 500,
        "artifacts_subdir": "farooq_fair_benchmark_medium",
    },
    "large": {
        "train_sizes": [200, 500],
        "validation_size": 200,
        "test_size": 300,
        "selection_seeds": [7, 11, 13, 21, 42],
        "final_seeds": [7, 11, 13, 21, 42, 77, 101, 123, 314, 2026],
        "bootstrap_iterations": 1000,
        "artifacts_subdir": "farooq_fair_benchmark",
    },
}

p = PRESETS[MODE]
ART = WORK / "artifacts" / p["artifacts_subdir"]
DATA_RAW = WORK / "data" / "raw"
CFG_PATH = WORK / "farooq_fair_benchmark.yaml"

# YAML mínimo do experimento (escritos no WORK)
CFG_TEXT = f"""
experiment:
  name: farooq_style_fair_benchmark
  horizon_hours: 24
  extreme_percentile: 0.90
  selection_seeds: {p['selection_seeds']}
  final_seeds: {p['final_seeds']}

project:
  random_seed: 42

data:
  dataset_id: 501
  station: Aotizhongxin
  target_column: PM2.5
  raw_dir: {DATA_RAW.as_posix()}
  purge_hours: 24

features:
  mode: farooq_stats
  source_columns:
    PM2.5: pm25
    TEMP: temperature
  stats: [min, max, median, variance]
  rolling_window_hours: 24
  rolling_min_periods: 12
  ffill_limit_hours: 3

split:
  train_fraction: 0.60
  validation_fraction: 0.20
  test_fraction: 0.20

sampling:
  train_sizes: {p['train_sizes']}
  validation_size: {p['validation_size']}
  test_size: {p['test_size']}
  evaluation_seed: 2026
  fixed_validation: true
  fixed_test: true

classical:
  logistic:
    C: [0.1, 1.0, 10.0]
    class_weight: [null, balanced]
  svm_linear:
    C: [0.1, 1.0, 10.0]
    class_weight: [null, balanced]
  svm_rbf:
    C: [0.1, 1.0, 10.0]
    gamma: [scale, 0.1, 1.0]
    class_weight: [null, balanced]

quantum:
  feature_map: ZZFeatureMap
  qubits: 2
  reps: [1, 2]
  angular_scalers: [minmax_0_1, minmax_0_pi]
  entanglement: linear
  C: [0.1, 1.0, 10.0]
  class_weight: [null, balanced]
  cache_kernels: true

threshold:
  metric: f2
  select_on: validation

evaluation:
  primary_metric: average_precision
  bootstrap_iterations: {p['bootstrap_iterations']}
  bootstrap_block_hours: 24

paths:
  artifacts_dir: {ART.as_posix()}
"""
CFG_PATH.write_text(CFG_TEXT, encoding="utf-8")
DATA_RAW.mkdir(parents=True, exist_ok=True)
ART.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO if VERBOSE else logging.WARNING,
    format="%(levelname)s %(name)s: %(message)s",
)

print("=" * 60)
print(f"MODE={MODE}  STAGE={STAGE}  SKIP_QUANTUM={SKIP_QUANTUM}")
print(f"train={p['train_sizes']} val/test={p['validation_size']}/{p['test_size']}")
print(f"ART={ART}")
print(f"CFG={CFG_PATH}")
print("=" * 60)


## 4. Patch de paths do pacote (WORK em vez da raiz do repo)


In [ ]:
# project_root() aponta para WORK (pacote embutido em WORK/src/...)
from qml_air_quality import config as _cfg
print("project_root() ->", _cfg.project_root())
assert _cfg.project_root().resolve() == WORK.resolve()


## 5. Executar experimento


In [ ]:
from qml_air_quality.experiments.fair_benchmark import run_fair_benchmark

result = run_fair_benchmark(
    CFG_PATH,
    stage=STAGE,
    skip_quantum=SKIP_QUANTUM,
)

if result.get("selection") is not None:
    print("selection rows:", len(result["selection"]))
if result.get("metrics") is not None:
    print("final metric rows:", len(result["metrics"]))
if result.get("plots"):
    print("plots:", len(result["plots"]))
print("done.")


## 6. Tabelas


In [ ]:
from IPython.display import display

summary = pd.read_csv(ART / "final_metrics_summary.csv")
display(summary.sort_values(["train_size", "average_precision_mean"], ascending=[True, False]))

boot_path = ART / "pairwise_bootstrap.csv"
if boot_path.exists() and boot_path.stat().st_size > 1:
    boot = pd.read_csv(boot_path)
    cols = [c for c in ["comparison", "train_size", "label", "delta_auprc_mean", "delta_auprc_ci_low", "delta_auprc_ci_high", "conclusion"] if c in boot.columns]
    display(boot[cols] if cols else boot)
else:
    print("pairwise_bootstrap vazio (rode com SKIP_QUANTUM=False).")

print((ART / "final_report.md").read_text(encoding="utf-8"))


## 7. Painel visual


In [ ]:
from IPython.display import Image, display
from qml_air_quality.reporting.fair_benchmark_plots import generate_fair_benchmark_plots

plot_paths = generate_fair_benchmark_plots(ART)
print(f"{len(plot_paths)} plots → {ART / 'plots'}")
for path in plot_paths:
    print(path.name)
    display(Image(filename=str(path)))
